In [ ]:
# ===== CELL 1: Setup + download + inspect =====
import os, glob, subprocess
from collections import Counter

# 1. Check GPU
print("GPU:", subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU - change runtime type")

# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# 3. Create project folders on Drive
PROJECT = "/content/drive/MyDrive/egg_project"
for sub in ["results/sam_masks", "results/sam_overlays", "results/classical_masks", "experiments", "video"]:
    os.makedirs(PROJECT + "/" + sub, exist_ok=True)
print("Project folder:", PROJECT)

# 4. Download dataset (key typed in a hidden box, not saved in the notebook)
!pip install -q roboflow
from roboflow import Roboflow
from getpass import getpass

DATA_DIR = "/content/egg_dataset"
if os.path.exists(DATA_DIR) and len(os.listdir(DATA_DIR)) > 0:
    print("Dataset already downloaded:", DATA_DIR)
else:
    api_key = getpass("Paste Roboflow API key: ").strip()
    rf = Roboflow(api_key=api_key)
    project = rf.workspace("fire-bf7qe").project("egg-vzhtr")
    version = project.version(1)
    dataset = version.download("folder", location=DATA_DIR)

# 5. Show folder structure
print("\n=== FOLDERS ===")
for root, dirs, files in os.walk(DATA_DIR):
    depth = root.replace(DATA_DIR, "").count(os.sep)
    print("  " * depth + os.path.basename(root) + "/  (" + str(len(files)) + " files)")

# 6. Count images and check sizes
import cv2
exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
images = []
for p in glob.glob(DATA_DIR + "/**/*", recursive=True):
    if p.lower().endswith(exts):
        images.append(p)
images.sort()

print("\n=== IMAGES ===")
print("Total images:", len(images))
print("File types:", Counter(os.path.splitext(p)[1].lower() for p in images))
print("Per folder:", Counter(os.path.basename(os.path.dirname(p)) for p in images))

sizes = Counter()
for p in images[:50]:
    img = cv2.imread(p)
    if img is not None:
        sizes[img.shape[:2]] += 1
print("Sizes (height, width) in first 50:", dict(sizes))

# 7. Other files (annotations, README, license)
print("\n=== NON-IMAGE FILES ===")
for p in glob.glob(DATA_DIR + "/**/*", recursive=True):
    if os.path.isfile(p) and not p.lower().endswith(exts):
        print(p)

# 8. Show 4 sample images
import matplotlib.pyplot as plt
plt.figure(figsize=(14, 4))
for i in range(min(4, len(images))):
    plt.subplot(1, 4, i + 1)
    plt.imshow(cv2.cvtColor(cv2.imread(images[i]), cv2.COLOR_BGR2RGB))
    plt.title(os.path.basename(images[i])[:25], fontsize=7)
    plt.axis("off")
plt.show()

In [ ]:
# ===== CELL 2: Read licence + view images =====
import glob, os, cv2
import matplotlib.pyplot as plt

DATA_DIR = "/content/egg_dataset"

# 1. Print the README files (licence info is in here)
for name in ["README.dataset.txt", "README.roboflow.txt"]:
    path = DATA_DIR + "/" + name
    print("=====", name, "=====")
    print(open(path).read()[:1500])
    print()

# 2. Collect all images
exts = (".jpg", ".jpeg", ".png")
images = []
for p in glob.glob(DATA_DIR + "/**/*", recursive=True):
    if p.lower().endswith(exts):
        images.append(p)
images.sort()

# 3. Show 8 images spread across the dataset
step = len(images) // 8
chosen = images[::step][:8]
plt.figure(figsize=(16, 8))
for i in range(len(chosen)):
    plt.subplot(2, 4, i + 1)
    plt.imshow(cv2.cvtColor(cv2.imread(chosen[i]), cv2.COLOR_BGR2RGB))
    plt.title(chosen[i].split("egg_dataset/")[1][:30], fontsize=8)
    plt.axis("off")
plt.show()

# 4. Show the 2 'Unlabeled' images
unl = []
for p in images:
    if "Unlabeled" in p:
        unl.append(p)
plt.figure(figsize=(8, 4))
for i in range(len(unl)):
    plt.subplot(1, 2, i + 1)
    plt.imshow(cv2.cvtColor(cv2.imread(unl[i]), cv2.COLOR_BGR2RGB))
    plt.title("Unlabeled", fontsize=8)
    plt.axis("off")
plt.show()

In [ ]:
# ===== CELL 3: Download + inspect the video =====
import os, subprocess, cv2
import matplotlib.pyplot as plt

PROJECT = "/content/drive/MyDrive/egg_project"
VIDEO_PATH = PROJECT + "/video/input.mp4"

!pip install -q yt-dlp

if os.path.exists(VIDEO_PATH):
    print("Video already downloaded.")
else:
    !yt-dlp -f "best[height<=720][ext=mp4]/best[ext=mp4]/best" -o "/content/input.mp4" "https://www.youtube.com/watch?v=Y4JyV94y2_A"
    if os.path.exists("/content/input.mp4"):
        !cp /content/input.mp4 "{VIDEO_PATH}"

if os.path.exists(VIDEO_PATH):
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print("Resolution:", w, "x", h)
    print("FPS:", round(fps, 1), "| Frames:", frames, "| Length (s):", round(frames / fps, 1))

    # show 3 frames: start, middle, end
    plt.figure(figsize=(15, 4))
    for i, pos in enumerate([0.1, 0.5, 0.9]):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frames * pos))
        ok, frame = cap.read()
        if ok:
            plt.subplot(1, 3, i + 1)
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.title("at " + str(int(pos * 100)) + "% of video")
            plt.axis("off")
    plt.show()
    cap.release()
else:
    print("DOWNLOAD FAILED. Download the video on your laptop and upload it to Drive at:", VIDEO_PATH)

In [ ]:
# ===== CELL 3b: Inspect the uploaded video =====
import os, cv2
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/drive/MyDrive/egg_project/video/input.mp4"

if not os.path.exists(VIDEO_PATH):
    print("Video not found at", VIDEO_PATH, "- check the upload and the file name.")
else:
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print("Resolution:", w, "x", h)
    print("FPS:", round(fps, 1), "| Frames:", frames, "| Length (s):", round(frames / fps, 1))

    plt.figure(figsize=(15, 4))
    for i, pos in enumerate([0.1, 0.5, 0.9]):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frames * pos))
        ok, frame = cap.read()
        if ok:
            plt.subplot(1, 3, i + 1)
            plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            plt.title("at " + str(int(pos * 100)) + "% of video")
            plt.axis("off")
    plt.show()
    cap.release()

In [ ]:
# ===== CELL 4: Install SAM + test on 6 images =====
import os, glob, subprocess, sys
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt

# 1. Install SAM and download the ViT-B checkpoint (only if missing)
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
CKPT = "/content/sam_vit_b_01ec64.pth"
if not os.path.exists(CKPT):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -O /content/sam_vit_b_01ec64.pth
print("Checkpoint size (MB):", round(os.path.getsize(CKPT) / 1e6))

# 2. Load the model onto the GPU
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
sam = sam_model_registry["vit_b"](checkpoint=CKPT)
sam.to(device)

# 3. Mask generator (settings: how many points to try, how strict to be)
mask_generator = SamAutomaticMaskGenerator(
    sam,
    points_per_side=32,
    pred_iou_thresh=0.88,
    stability_score_thresh=0.92,
    min_mask_region_area=200,
)

# 4. Get the image list and pick 6 spread across the dataset
DATA_DIR = "/content/egg_dataset"
images = sorted(glob.glob(DATA_DIR + "/**/*.jpg", recursive=True))
step = len(images) // 6
test_images = images[::step][:6]

# 5. Run SAM on each and show numbered regions
for path in test_images:
    img_bgr = cv2.imread(path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    masks = mask_generator.generate(img_rgb)
    masks = sorted(masks, key=lambda m: m["area"], reverse=True)

    # draw every region in a random colour with its number
    overlay = img_rgb.copy()
    rng = np.random.RandomState(0)
    for i in range(len(masks)):
        colour = rng.randint(0, 255, 3)
        overlay[masks[i]["segmentation"]] = (0.5 * overlay[masks[i]["segmentation"]] + 0.5 * colour).astype(np.uint8)
    for i in range(len(masks)):
        x, y, bw, bh = masks[i]["bbox"]
        cv2.putText(overlay, str(i), (int(x + bw / 2), int(y + bh / 2)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1); plt.imshow(img_rgb); plt.title(path.split("egg_dataset/")[1]); plt.axis("off")
    plt.subplot(1, 2, 2); plt.imshow(overlay); plt.title(str(len(masks)) + " regions found"); plt.axis("off")
    plt.show()

    print("Regions for", os.path.basename(path)[:30])
    for i in range(min(10, len(masks))):
        print("  #", i, "area:", masks[i]["area"], "| quality:", round(masks[i]["predicted_iou"], 2))

In [ ]:
# ===== CELL 5 (v2): Test the egg-picking rule on 8 images =====
import glob, os
import numpy as np
import cv2
import matplotlib.pyplot as plt

if "mask_generator" not in globals():
    raise RuntimeError("Run Cell 4 first (it loads SAM).")

# ---- Rule settings (hypothesis v2, we will tune these) ----
MIN_AREA_FRAC = 0.02    # at least 2% of the image
MAX_AREA_FRAC = 0.90    # at most 90% (close-up eggs can be big)
MIN_SOLIDITY  = 0.88    # smooth blob (area / convex hull area)
MAX_ASPECT    = 2.5     # long side / short side
MAX_EDGE_COV  = 0.50    # background wraps around the image border
MAX_SAT       = 130     # eggs are whitish -> low saturation (0-255)
MIN_VAL       = 100     # ...and fairly bright (0-255)

def region_features(mask, hsv, img_area):
    """Return a dict of simple numbers describing one SAM region."""
    mask_u8 = mask.astype(np.uint8)
    h, w = mask_u8.shape
    area = int(mask_u8.sum())
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None
    big = max(contours, key=cv2.contourArea)
    hull = cv2.convexHull(big)
    hull_area = cv2.contourArea(hull)
    solidity = cv2.contourArea(big) / hull_area if hull_area > 0 else 0
    (cx, cy), (rw, rh), angle = cv2.minAreaRect(big)
    short_side = min(rw, rh)
    aspect = max(rw, rh) / short_side if short_side > 0 else 99
    # how much of the image border does this region cover?
    border_pixels = mask_u8[0, :].sum() + mask_u8[-1, :].sum() + mask_u8[:, 0].sum() + mask_u8[:, -1].sum()
    edge_cov = border_pixels / (2 * (h + w))
    sat = float(hsv[:, :, 1][mask].mean())
    val = float(hsv[:, :, 2][mask].mean())
    return {"area_frac": area / img_area, "solidity": solidity, "aspect": aspect,
            "edge": edge_cov, "sat": sat, "val": val}

def is_egg(f):
    """Apply the rule. Returns True or False."""
    if f is None:
        return False
    if f["area_frac"] < MIN_AREA_FRAC or f["area_frac"] > MAX_AREA_FRAC:
        return False
    if f["solidity"] < MIN_SOLIDITY:
        return False
    if f["aspect"] > MAX_ASPECT:
        return False
    if f["edge"] > MAX_EDGE_COV:
        return False
    if f["sat"] > MAX_SAT or f["val"] < MIN_VAL:
        return False
    return True

def fill_holes(mask_bool):
    """Fill small holes (dirt spots, reflections) inside the mask."""
    m = mask_bool.astype(np.uint8)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(m)
    cv2.drawContours(filled, contours, -1, 1, thickness=-1)
    return filled.astype(bool)

# ---- Same 8 test images as before ----
DATA_DIR = "/content/egg_dataset"
all_images = sorted(glob.glob(DATA_DIR + "/**/*.jpg", recursive=True))
unlabeled = [p for p in all_images if "Unlabeled" in p]
test_set = list(test_images) + unlabeled

for path in test_set:
    img_rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    img_area = img_rgb.shape[0] * img_rgb.shape[1]

    masks = mask_generator.generate(img_rgb)
    masks = sorted(masks, key=lambda m: m["area"], reverse=True)

    union = np.zeros(img_rgb.shape[:2], dtype=bool)
    kept = 0
    print("\n", os.path.basename(path)[:35], "| regions:", len(masks))
    print("  id  area%  solid  aspect  edge  sat  val  -> egg?")
    for i in range(len(masks)):
        f = region_features(masks[i]["segmentation"], hsv, img_area)
        keep = is_egg(f)
        if keep:
            union = union | masks[i]["segmentation"]
            kept += 1
        if i < 12 and f is not None:
            print("  ", i, round(100 * f["area_frac"], 1), round(f["solidity"], 2),
                  round(f["aspect"], 2), round(f["edge"], 2), int(f["sat"]), int(f["val"]), "->", keep)

    final_mask = fill_holes(union)
    n_blobs, _ = cv2.connectedComponents(final_mask.astype(np.uint8))
    print("  regions kept:", kept, "| separate blobs in final mask:", n_blobs - 1)

    overlay = img_rgb.copy()
    overlay[final_mask] = (0.5 * overlay[final_mask] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1); plt.imshow(img_rgb); plt.title(path.split("egg_dataset/")[1][:30], fontsize=8); plt.axis("off")
    plt.subplot(1, 2, 2); plt.imshow(overlay); plt.title("green = picked as egg", fontsize=8); plt.axis("off")
    plt.show()

In [ ]:
# ===== CELL 5 (v3): egg-picking rule, updated numbers =====
import glob, os
import numpy as np
import cv2
import matplotlib.pyplot as plt

if "mask_generator" not in globals():
    raise RuntimeError("Run Cell 4 first (it loads SAM).")

# ---- Rule settings v3 ----
MIN_AREA_FRAC = 0.02
MAX_AREA_FRAC = 0.90
MIN_SOLIDITY  = 0.88
MAX_ASPECT    = 2.5
MAX_EDGE_COV  = 0.20    # was 0.50
MAX_SAT       = 70      # was 130
MIN_VAL       = 170     # was 100

def region_features(mask, hsv, img_area):
    mask_u8 = mask.astype(np.uint8)
    h, w = mask_u8.shape
    area = int(mask_u8.sum())
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None
    big = max(contours, key=cv2.contourArea)
    hull_area = cv2.contourArea(cv2.convexHull(big))
    solidity = cv2.contourArea(big) / hull_area if hull_area > 0 else 0
    (cx, cy), (rw, rh), angle = cv2.minAreaRect(big)
    short_side = min(rw, rh)
    aspect = max(rw, rh) / short_side if short_side > 0 else 99
    border = mask_u8[0, :].sum() + mask_u8[-1, :].sum() + mask_u8[:, 0].sum() + mask_u8[:, -1].sum()
    edge_cov = border / (2 * (h + w))
    return {"area_frac": area / img_area, "solidity": solidity, "aspect": aspect,
            "edge": edge_cov, "sat": float(hsv[:, :, 1][mask].mean()),
            "val": float(hsv[:, :, 2][mask].mean())}

def is_egg(f):
    if f is None:
        return False
    if f["area_frac"] < MIN_AREA_FRAC or f["area_frac"] > MAX_AREA_FRAC:
        return False
    if f["solidity"] < MIN_SOLIDITY or f["aspect"] > MAX_ASPECT:
        return False
    if f["edge"] > MAX_EDGE_COV:
        return False
    if f["sat"] > MAX_SAT or f["val"] < MIN_VAL:
        return False
    return True

def fill_holes(mask_bool):
    m = mask_bool.astype(np.uint8)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(m)
    cv2.drawContours(filled, contours, -1, 1, thickness=-1)
    return filled.astype(bool)

# cache so SAM runs only once per image
if "sam_cache" not in globals():
    sam_cache = {}

DATA_DIR = "/content/egg_dataset"
all_images = sorted(glob.glob(DATA_DIR + "/**/*.jpg", recursive=True))
unlabeled = [p for p in all_images if "Unlabeled" in p]
test_set = list(test_images) + unlabeled

plt.figure(figsize=(20, 10))
for n in range(len(test_set)):
    path = test_set[n]
    img_rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    img_area = img_rgb.shape[0] * img_rgb.shape[1]

    if path not in sam_cache:
        sam_cache[path] = mask_generator.generate(img_rgb)
    masks = sorted(sam_cache[path], key=lambda m: m["area"], reverse=True)

    union = np.zeros(img_rgb.shape[:2], dtype=bool)
    print("\n#" + str(n + 1), os.path.basename(path)[:30], "| kept regions:")
    for i in range(len(masks)):
        f = region_features(masks[i]["segmentation"], hsv, img_area)
        if is_egg(f):
            union = union | masks[i]["segmentation"]
            print("   id", i, "area%", round(100 * f["area_frac"], 1), "sat", int(f["sat"]), "val", int(f["val"]))
    final_mask = fill_holes(union)
    n_blobs, _ = cv2.connectedComponents(final_mask.astype(np.uint8))
    print("   blobs in final mask:", n_blobs - 1)

    overlay = img_rgb.copy()
    overlay[final_mask] = (0.5 * overlay[final_mask] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)
    plt.subplot(2, 4, n + 1)
    plt.imshow(overlay)
    plt.title("#" + str(n + 1) + "  blobs: " + str(n_blobs - 1))
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ===== CELL 5c: Unlabeled images with a pixel grid =====
import glob
import cv2
import matplotlib.pyplot as plt

unl = sorted(glob.glob("/content/egg_dataset/**/Unlabeled/*.jpg", recursive=True))
plt.figure(figsize=(18, 9))
for i in range(len(unl)):
    plt.subplot(1, 2, i + 1)
    plt.imshow(cv2.cvtColor(cv2.imread(unl[i]), cv2.COLOR_BGR2RGB))
    plt.xticks(range(0, 641, 50))
    plt.yticks(range(0, 641, 50))
    plt.grid(True, color="yellow", alpha=0.5)
    plt.title(unl[i].split("/")[-1][:30])
plt.show()

In [ ]:
# ===== CELL 6: Task 1 - run on all images, save masks + overlays =====
import os, glob, csv
import numpy as np
import cv2
import matplotlib.pyplot as plt

if "mask_generator" not in globals() or "is_egg" not in globals():
    raise RuntimeError("Run Cell 4 and Cell 5 (v3) first in this session.")

from segment_anything import SamPredictor
predictor = SamPredictor(sam)

PROJECT = "/content/drive/MyDrive/egg_project"
MASK_DIR = PROJECT + "/results/sam_masks"
OVER_DIR = PROJECT + "/results/sam_overlays"
LOG_PATH = PROJECT + "/results/sam_reference_log.csv"
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(OVER_DIR, exist_ok=True)

# Manual boxes (left, top, right, bottom) for the 2 Unlabeled images, in sorted-name order
MANUAL_BOXES = [[300, 250, 500, 400], [275, 275, 450, 400]]

DATA_DIR = "/content/egg_dataset"
all_images = sorted(glob.glob(DATA_DIR + "/**/*.jpg", recursive=True))
unlabeled_paths = [p for p in all_images if "Unlabeled" in p]
print("Total images:", len(all_images), "| manual-box images:", len(unlabeled_paths))
for i in range(len(unlabeled_paths)):
    print("  manual box", MANUAL_BOXES[i], "->", os.path.basename(unlabeled_paths[i])[:40])

if "sam_cache" not in globals():
    sam_cache = {}

rows = []
for n in range(len(all_images)):
    path = all_images[n]
    stem = os.path.splitext(os.path.basename(path))[0]
    mask_path = MASK_DIR + "/" + stem + ".png"
    over_path = OVER_DIR + "/" + stem + ".jpg"
    is_manual = path in unlabeled_paths
    method = "manual_box" if is_manual else "auto_rule"
    split_folder = path.split("egg_dataset/")[1].split("/")[0]

    if os.path.exists(mask_path) and os.path.exists(over_path):
        final_mask = cv2.imread(mask_path, 0) > 0          # resume: reuse saved mask
        status = "skipped (already saved)"
    else:
        img_rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
        img_area = img_rgb.shape[0] * img_rgb.shape[1]

        if is_manual:
            box = np.array(MANUAL_BOXES[unlabeled_paths.index(path)])
            predictor.set_image(img_rgb)
            out_masks, scores, _ = predictor.predict(box=box, multimask_output=False)
            union = out_masks[0]
        else:
            if path not in sam_cache:
                sam_cache[path] = mask_generator.generate(img_rgb)
            union = np.zeros(img_rgb.shape[:2], dtype=bool)
            for m in sam_cache[path]:
                f = region_features(m["segmentation"], hsv, img_area)
                if is_egg(f):
                    union = union | m["segmentation"]

        final_mask = fill_holes(union)
        cv2.imwrite(mask_path, final_mask.astype(np.uint8) * 255)

        overlay = img_rgb.copy()
        overlay[final_mask] = (0.5 * overlay[final_mask] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)
        cv2.imwrite(over_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
        status = "done"

    n_labels, _ = cv2.connectedComponents(final_mask.astype(np.uint8))
    rows.append({"image": stem, "roboflow_folder": split_folder, "method": method,
                 "blobs": n_labels - 1, "egg_area_pct": round(100 * final_mask.mean(), 2)})
    if (n + 1) % 10 == 0 or n == len(all_images) - 1:
        print("progress:", n + 1, "/", len(all_images), "| last:", status)

# Save the log
with open(LOG_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)
print("\nLog saved:", LOG_PATH)

# Summary
zero = [r["image"] for r in rows if r["blobs"] == 0]
print("Images with ZERO eggs found:", len(zero))
for z in zero:
    print("   ", z)
from collections import Counter
print("Blobs per image:", dict(sorted(Counter(r["blobs"] for r in rows).items())))

# Contact sheet: 12 random overlays (fixed seed) for a quick visual check
rng = np.random.RandomState(0)
pick = rng.choice(len(rows), 12, replace=False)
plt.figure(figsize=(20, 15))
for k in range(12):
    r = rows[pick[k]]
    ov = cv2.cvtColor(cv2.imread(OVER_DIR + "/" + r["image"] + ".jpg"), cv2.COLOR_BGR2RGB)
    plt.subplot(3, 4, k + 1)
    plt.imshow(ov)
    plt.title(r["image"][:22] + "  blobs:" + str(r["blobs"]), fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ===== CELL 5b: Video contact sheet (1 frame every 2 seconds) =====
import cv2
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/drive/MyDrive/egg_project/video/input.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

plt.figure(figsize=(20, 14))
count = 0
for sec in range(0, int(total / fps), 2):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(sec * fps))
    ok, frame = cap.read()
    if not ok:
        continue
    count += 1
    plt.subplot(5, 6, count)
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title(str(sec) + " s", fontsize=10)
    plt.axis("off")
plt.tight_layout()
plt.show()
cap.release()

In [ ]:
# ===== CELL 6b: Diagnose the zero-egg images and the many-blob images =====
import os, csv, glob
import numpy as np
import cv2
import matplotlib.pyplot as plt

if "region_features" not in globals() or "mask_generator" not in globals():
    raise RuntimeError("Run Cell 4 and Cell 5 (v3) first in this session.")

PROJECT = "/content/drive/MyDrive/egg_project"
LOG_PATH = PROJECT + "/results/sam_reference_log.csv"
OVER_DIR = PROJECT + "/results/sam_overlays"
DATA_DIR = "/content/egg_dataset"

all_images = sorted(glob.glob(DATA_DIR + "/**/*.jpg", recursive=True))
stem_to_path = {}
for p in all_images:
    stem_to_path[os.path.splitext(os.path.basename(p))[0]] = p
if "sam_cache" not in globals():
    sam_cache = {}

rows = list(csv.DictReader(open(LOG_PATH)))
zero_stems = [r["image"] for r in rows if r["blobs"] == "0"]
many_stems = [r["image"] for r in rows if int(r["blobs"]) >= 4]

def why_failed(f):
    """List every check this region fails."""
    reasons = []
    if f["area_frac"] < MIN_AREA_FRAC: reasons.append("too small")
    if f["area_frac"] > MAX_AREA_FRAC: reasons.append("too big")
    if f["solidity"] < MIN_SOLIDITY:   reasons.append("not smooth(" + str(round(f["solidity"], 2)) + ")")
    if f["aspect"] > MAX_ASPECT:       reasons.append("too long(" + str(round(f["aspect"], 1)) + ")")
    if f["edge"] > MAX_EDGE_COV:       reasons.append("touches edge(" + str(round(f["edge"], 2)) + ")")
    if f["sat"] > MAX_SAT:             reasons.append("too colourful(" + str(int(f["sat"])) + ")")
    if f["val"] < MIN_VAL:             reasons.append("too dark(" + str(int(f["val"])) + ")")
    return reasons

# ---- Part 1: zero-egg images ----
print("ZERO-EGG IMAGES: what the 3 biggest regions failed on\n")
plt.figure(figsize=(20, 14))
for n in range(len(zero_stems)):
    path = stem_to_path[zero_stems[n]]
    img_rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    img_area = img_rgb.shape[0] * img_rgb.shape[1]
    if path not in sam_cache:
        sam_cache[path] = mask_generator.generate(img_rgb)
    masks = sorted(sam_cache[path], key=lambda m: m["area"], reverse=True)

    print("Z" + str(n + 1), zero_stems[n][:28], "| regions:", len(masks))
    shown = 0
    for i in range(len(masks)):
        f = region_features(masks[i]["segmentation"], hsv, img_area)
        if f is None or f["area_frac"] < 0.02:
            continue
        print("    region", i, "area%", round(100 * f["area_frac"], 1), "->", why_failed(f))
        shown += 1
        if shown == 3:
            break

    plt.subplot(4, 4, n + 1)
    plt.imshow(img_rgb)
    plt.title("Z" + str(n + 1), fontsize=12)
    plt.axis("off")
plt.tight_layout()
plt.show()

# ---- Part 2: images with 4+ blobs ----
print("\nIMAGES WITH 4 OR MORE BLOBS:")
plt.figure(figsize=(14, 6))
for n in range(len(many_stems)):
    ov = cv2.cvtColor(cv2.imread(OVER_DIR + "/" + many_stems[n] + ".jpg"), cv2.COLOR_BGR2RGB)
    print("M" + str(n + 1), many_stems[n][:28])
    plt.subplot(1, max(1, len(many_stems)), n + 1)
    plt.imshow(ov)
    plt.title("M" + str(n + 1), fontsize=12)
    plt.axis("off")
plt.show()

In [ ]:
# ===== CELL 7: rule v4 (MIN_VAL 155) on all images, compare with v3 =====
import os, glob, csv
import numpy as np
import cv2
import matplotlib.pyplot as plt

if "mask_generator" not in globals() or "region_features" not in globals():
    raise RuntimeError("Run Cell 4 and Cell 5 (v3) first in this session.")

MIN_VAL = 155      # v4: was 170 (is_egg reads this global)

from segment_anything import SamPredictor
predictor = SamPredictor(sam)

PROJECT = "/content/drive/MyDrive/egg_project"
OLD_MASK = PROJECT + "/results/sam_masks"
NEW_MASK = PROJECT + "/results/sam_masks_v4"
NEW_OVER = PROJECT + "/results/sam_overlays_v4"
os.makedirs(NEW_MASK, exist_ok=True)
os.makedirs(NEW_OVER, exist_ok=True)

# Manual boxes: image name prefix -> list of boxes (left, top, right, bottom)
MANUAL = {
    "not_damaged_135_jpg": [[300, 250, 500, 400]],
    "not_damaged_145_jpg": [[275, 275, 450, 400]],
}

if "sam_cache" not in globals():
    sam_cache = {}
all_images = sorted(glob.glob("/content/egg_dataset/**/*.jpg", recursive=True))

changed = []      # (stem, old_blobs, new_blobs)
still_zero = []
rows = []
for n in range(len(all_images)):
    path = all_images[n]
    stem = os.path.splitext(os.path.basename(path))[0]
    img_rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    img_area = img_rgb.shape[0] * img_rgb.shape[1]

    # is this a manual image?
    boxes = None
    method = "auto_rule_v4"
    for key in MANUAL:
        if stem.startswith(key):
            boxes = MANUAL[key]
            method = "manual_box"

    if boxes is not None:
        predictor.set_image(img_rgb)
        union = np.zeros(img_rgb.shape[:2], dtype=bool)
        for b in boxes:
            out, scores, _ = predictor.predict(box=np.array(b), multimask_output=False)
            union = union | out[0]
    else:
        if path not in sam_cache:
            sam_cache[path] = mask_generator.generate(img_rgb)
        union = np.zeros(img_rgb.shape[:2], dtype=bool)
        for m in sam_cache[path]:
            f = region_features(m["segmentation"], hsv, img_area)
            if is_egg(f):
                union = union | m["segmentation"]

    final_mask = fill_holes(union)
    cv2.imwrite(NEW_MASK + "/" + stem + ".png", final_mask.astype(np.uint8) * 255)
    overlay = img_rgb.copy()
    overlay[final_mask] = (0.5 * overlay[final_mask] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)
    cv2.imwrite(NEW_OVER + "/" + stem + ".jpg", cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    new_blobs = cv2.connectedComponents(final_mask.astype(np.uint8))[0] - 1
    old = cv2.imread(OLD_MASK + "/" + stem + ".png", 0) > 0
    old_blobs = cv2.connectedComponents(old.astype(np.uint8))[0] - 1
    area_change = abs(final_mask.mean() - old.mean()) * 100
    if new_blobs != old_blobs or area_change > 2:
        changed.append((stem, old_blobs, new_blobs))
    if new_blobs == 0:
        still_zero.append(stem)
    rows.append({"image": stem, "method": method, "blobs": new_blobs,
                 "egg_area_pct": round(100 * final_mask.mean(), 2)})

with open(PROJECT + "/results/sam_reference_log_v4.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

print("Images changed vs v3:", len(changed))
for i in range(len(changed)):
    print("  C" + str(i + 1), changed[i][0][:30], "| blobs", changed[i][1], "->", changed[i][2])
print("\nStill ZERO eggs:", len(still_zero))
for s in still_zero:
    print("  ", s[:35])

# grid of changed images (first 16)
show = changed[:16]
plt.figure(figsize=(20, 5 * ((len(show) + 3) // 4)))
for i in range(len(show)):
    ov = cv2.cvtColor(cv2.imread(NEW_OVER + "/" + show[i][0] + ".jpg"), cv2.COLOR_BGR2RGB)
    plt.subplot((len(show) + 3) // 4, 4, i + 1)
    plt.imshow(ov)
    plt.title("C" + str(i + 1), fontsize=12)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ===== CELL 7b: grids for the manual-box images =====
import glob, os
import cv2
import matplotlib.pyplot as plt

all_images = sorted(glob.glob("/content/egg_dataset/**/*.jpg", recursive=True))
wanted = [("Z2", "not_damaged_113_jpg"), ("Z3", "not_damaged_118_jpg"),
          ("Z5", "not_damaged_12_jpg"), ("Z8", "not_damaged_20_jpg"),
          ("M2", "not_damaged_42_jpg")]

plt.figure(figsize=(24, 14))
for i in range(len(wanted)):
    label, prefix = wanted[i]
    for p in all_images:
        if os.path.basename(p).startswith(prefix):
            plt.subplot(2, 3, i + 1)
            plt.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
            plt.xticks(range(0, 641, 50))
            plt.yticks(range(0, 641, 50))
            plt.grid(True, color="yellow", alpha=0.5)
            plt.title(label, fontsize=14)
            break
plt.tight_layout()
plt.show()

In [ ]:
# ===== CELL 8: reference masks v5 (strict first, relax only if nothing found) =====
import os, glob, csv
import numpy as np
import cv2
import matplotlib.pyplot as plt

if "mask_generator" not in globals() or "region_features" not in globals():
    raise RuntimeError("Run Cell 4 and Cell 5 (v3) first in this session.")

from segment_anything import SamPredictor
predictor = SamPredictor(sam)

PROJECT = "/content/drive/MyDrive/egg_project"
V4_MASK = PROJECT + "/results/sam_masks_v4"
NEW_MASK = PROJECT + "/results/sam_masks_v5"
NEW_OVER = PROJECT + "/results/sam_overlays_v5"
os.makedirs(NEW_MASK, exist_ok=True)
os.makedirs(NEW_OVER, exist_ok=True)

# Manual boxes: image name prefix -> list of boxes (left, top, right, bottom)
MANUAL = {
    "not_damaged_135_jpg": [[300, 250, 500, 400]],
    "not_damaged_145_jpg": [[275, 275, 450, 400]],
    "not_damaged_113_jpg": [[350, 0, 550, 275], [300, 225, 475, 400]],   # 2 eggs
    "not_damaged_118_jpg": [[200, 225, 475, 500]],
    "not_damaged_12_jpg":  [[200, 200, 425, 450]],
    "not_damaged_20_jpg":  [[150, 125, 350, 300]],
    "not_damaged_42_jpg":  [[125, 0, 350, 150]],
}

def is_egg_v5(f, min_val):
    """Same checks as before, but the brightness limit is a parameter."""
    if f is None:
        return False
    if f["area_frac"] < MIN_AREA_FRAC or f["area_frac"] > MAX_AREA_FRAC:
        return False
    if f["solidity"] < MIN_SOLIDITY or f["aspect"] > MAX_ASPECT:
        return False
    if f["edge"] > MAX_EDGE_COV:
        return False
    if f["sat"] > MAX_SAT or f["val"] < min_val:
        return False
    return True

def pick_union(masks, hsv, img_area, min_val):
    """Merge all regions that pass the rule. Also return how many passed."""
    union = np.zeros(hsv.shape[:2], dtype=bool)
    count = 0
    for m in masks:
        f = region_features(m["segmentation"], hsv, img_area)
        if is_egg_v5(f, min_val):
            union = union | m["segmentation"]
            count = count + 1
    return union, count

if "sam_cache" not in globals():
    sam_cache = {}
all_images = sorted(glob.glob("/content/egg_dataset/**/*.jpg", recursive=True))

changed = []
still_zero = []
rows = []
for n in range(len(all_images)):
    path = all_images[n]
    stem = os.path.splitext(os.path.basename(path))[0]
    img_rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    img_area = img_rgb.shape[0] * img_rgb.shape[1]

    boxes = None
    for key in MANUAL:
        if stem.startswith(key):
            boxes = MANUAL[key]

    if boxes is not None:
        method = "manual_box"
        predictor.set_image(img_rgb)
        union = np.zeros(img_rgb.shape[:2], dtype=bool)
        for b in boxes:
            out, scores, _ = predictor.predict(box=np.array(b), multimask_output=False)
            union = union | out[0]
    else:
        if path not in sam_cache:
            sam_cache[path] = mask_generator.generate(img_rgb)
        masks = sam_cache[path]
        union, count = pick_union(masks, hsv, img_area, 170)       # step 1: strict
        method = "auto_strict"
        if count == 0:
            union, count = pick_union(masks, hsv, img_area, 155)   # step 2: relaxed
            method = "auto_relaxed"

    final_mask = fill_holes(union)
    cv2.imwrite(NEW_MASK + "/" + stem + ".png", final_mask.astype(np.uint8) * 255)
    overlay = img_rgb.copy()
    overlay[final_mask] = (0.5 * overlay[final_mask] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)
    cv2.imwrite(NEW_OVER + "/" + stem + ".jpg", cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    new_blobs = cv2.connectedComponents(final_mask.astype(np.uint8))[0] - 1
    old = cv2.imread(V4_MASK + "/" + stem + ".png", 0) > 0
    old_blobs = cv2.connectedComponents(old.astype(np.uint8))[0] - 1
    if new_blobs != old_blobs or abs(final_mask.mean() - old.mean()) * 100 > 2:
        changed.append((stem, old_blobs, new_blobs))
    if new_blobs == 0:
        still_zero.append(stem)
    rows.append({"image": stem, "method": method, "blobs": new_blobs,
                 "egg_area_pct": round(100 * final_mask.mean(), 2)})

with open(PROJECT + "/results/sam_reference_log_v5.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)

from collections import Counter
print("Methods used:", dict(Counter(r["method"] for r in rows)))
print("\nChanged vs v4:", len(changed))
for i in range(len(changed)):
    print("  D" + str(i + 1), changed[i][0][:30], "| blobs", changed[i][1], "->", changed[i][2])
print("\nStill ZERO eggs:", len(still_zero))
for s in still_zero:
    print("  ", s[:35])

show = changed[:20]
if len(show) > 0:
    plt.figure(figsize=(20, 5 * ((len(show) + 3) // 4)))
    for i in range(len(show)):
        ov = cv2.cvtColor(cv2.imread(NEW_OVER + "/" + show[i][0] + ".jpg"), cv2.COLOR_BGR2RGB)
        plt.subplot((len(show) + 3) // 4, 4, i + 1)
        plt.imshow(ov)
        plt.title("D" + str(i + 1), fontsize=12)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# ===== CELL 8b: grids for the remaining problem images =====
import glob, os
import cv2
import matplotlib.pyplot as plt

all_images = sorted(glob.glob("/content/egg_dataset/**/*.jpg", recursive=True))
wanted = [("Z4", "not_damaged_124_jpg"), ("Z10", "not_damaged_39_jpg"),
          ("Z11", "not_damaged_44_jpg"), ("Z13", "not_damaged_119_jpg"),
          ("Z14", "not_damaged_11_jpg"), ("C8", "not_damaged_117_jpg")]

plt.figure(figsize=(24, 14))
for i in range(len(wanted)):
    label, prefix = wanted[i]
    for p in all_images:
        if os.path.basename(p).startswith(prefix):
            plt.subplot(2, 3, i + 1)
            plt.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
            plt.xticks(range(0, 641, 50))
            plt.yticks(range(0, 641, 50))
            plt.grid(True, color="yellow", alpha=0.5)
            plt.title(label, fontsize=14)
            break
plt.tight_layout()
plt.show()

In [ ]:
# ===== CELL 8c: v5 overlays with pixel grid for the problem images =====
import glob, os
import cv2
import matplotlib.pyplot as plt

PROJECT = "/content/drive/MyDrive/egg_project"
OVER_DIR = PROJECT + "/results/sam_overlays_v5"

wanted = [("D3", "not_damaged_103_jpg"), ("D6", "not_damaged_113_jpg"),
          ("D7", "not_damaged_117_jpg"), ("D11", "not_damaged_142_jpg"),
          ("D15", "not_damaged_133_jpg"), ("D16", "not_damaged_140_jpg"),
          ("Z4", "not_damaged_124_jpg"), ("Z10", "not_damaged_39_jpg"),
          ("Z11", "not_damaged_44_jpg"), ("Z13", "not_damaged_119_jpg"),
          ("Z14", "not_damaged_11_jpg")]

files = sorted(glob.glob(OVER_DIR + "/*.jpg"))

def show_group(group):
    plt.figure(figsize=(24, 14))
    for i in range(len(group)):
        label, prefix = group[i]
        for p in files:
            if os.path.basename(p).startswith(prefix):
                plt.subplot(2, 3, i + 1)
                plt.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
                plt.xticks(range(0, 641, 50))
                plt.yticks(range(0, 641, 50))
                plt.grid(True, color="yellow", alpha=0.5)
                plt.title(label, fontsize=14)
                break
    plt.tight_layout()
    plt.show()

show_group(wanted[:6])
show_group(wanted[6:])

In [ ]:
# ===== CELL 9: FINAL reference masks (auto rule + manual boxes) =====
import os, glob, csv
import numpy as np
import cv2
import matplotlib.pyplot as plt
from collections import Counter

if "mask_generator" not in globals() or "region_features" not in globals():
    raise RuntimeError("Run Cell 4 and Cell 5 (v3) first in this session.")

from segment_anything import SamPredictor
predictor = SamPredictor(sam)

PROJECT = "/content/drive/MyDrive/egg_project"
FINAL_MASK = PROJECT + "/results/reference_masks"
FINAL_OVER = PROJECT + "/results/reference_overlays"
os.makedirs(FINAL_MASK, exist_ok=True)
os.makedirs(FINAL_OVER, exist_ok=True)

# Boxes are (left, top, right, bottom)
# These images use ONLY boxes (no automatic rule)
MANUAL_ONLY = {
    "not_damaged_135_jpg": [[300, 250, 500, 400]],
    "not_damaged_145_jpg": [[275, 275, 450, 400]],
    "not_damaged_113_jpg": [[350, 0, 550, 275], [300, 225, 475, 400], [0, 350, 275, 550]],
    "not_damaged_118_jpg": [[200, 225, 475, 500]],
    "not_damaged_12_jpg":  [[200, 200, 425, 450]],
    "not_damaged_20_jpg":  [[150, 125, 350, 300]],
    "not_damaged_42_jpg":  [[125, 0, 350, 150]],
    "not_damaged_124_jpg": [[300, 150, 500, 400]],
    "not_damaged_39_jpg":  [[350, 150, 550, 350]],
    "not_damaged_44_jpg":  [[0, 50, 600, 400]],
    "not_damaged_119_jpg": [[100, 200, 400, 550]],
    "not_damaged_11_jpg":  [[150, 125, 400, 350]],
}
# These images use the automatic rule, PLUS an extra box for the missed egg
EXTRA_BOXES = {
    "not_damaged_103_jpg": [[150, 50, 450, 250]],
    "not_damaged_117_jpg": [[250, 100, 525, 300]],
    "not_damaged_142_jpg": [[75, 250, 325, 450]],
    "not_damaged_133_jpg": [[350, 200, 640, 375]],
    "not_damaged_140_jpg": [[300, 400, 550, 600]],
}

def is_egg_v5(f, min_val):
    if f is None:
        return False
    if f["area_frac"] < MIN_AREA_FRAC or f["area_frac"] > MAX_AREA_FRAC:
        return False
    if f["solidity"] < MIN_SOLIDITY or f["aspect"] > MAX_ASPECT:
        return False
    if f["edge"] > MAX_EDGE_COV:
        return False
    if f["sat"] > MAX_SAT or f["val"] < min_val:
        return False
    return True

def pick_union(masks, hsv, img_area, min_val):
    union = np.zeros(hsv.shape[:2], dtype=bool)
    count = 0
    for m in masks:
        f = region_features(m["segmentation"], hsv, img_area)
        if is_egg_v5(f, min_val):
            union = union | m["segmentation"]
            count = count + 1
    return union, count

def biggest_piece(mask_bool):
    """Keep only the largest connected piece (removes stray specks)."""
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask_bool.astype(np.uint8))
    if n <= 1:
        return mask_bool
    best = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return labels == best

def box_mask(box, h, w):
    """SAM mask for one box, cut to the box, biggest piece only."""
    x1 = max(0, box[0]); y1 = max(0, box[1])
    x2 = min(w, box[2]); y2 = min(h, box[3])
    out, scores, _ = predictor.predict(box=np.array([x1, y1, x2, y2]), multimask_output=False)
    m = out[0]
    clipped = np.zeros_like(m)
    clipped[y1:y2, x1:x2] = m[y1:y2, x1:x2]
    return biggest_piece(clipped)

if "sam_cache" not in globals():
    sam_cache = {}
all_images = sorted(glob.glob("/content/egg_dataset/**/*.jpg", recursive=True))

rows = []
for n in range(len(all_images)):
    path = all_images[n]
    stem = os.path.splitext(os.path.basename(path))[0]
    img_rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    h, w = img_rgb.shape[:2]
    img_area = h * w

    only_boxes = None
    extra_boxes = None
    for key in MANUAL_ONLY:
        if stem.startswith(key):
            only_boxes = MANUAL_ONLY[key]
    for key in EXTRA_BOXES:
        if stem.startswith(key):
            extra_boxes = EXTRA_BOXES[key]

    if only_boxes is not None:
        method = "manual_box"
        predictor.set_image(img_rgb)
        union = np.zeros((h, w), dtype=bool)
        for b in only_boxes:
            union = union | box_mask(b, h, w)
    else:
        if path not in sam_cache:
            sam_cache[path] = mask_generator.generate(img_rgb)
        masks = sam_cache[path]
        union, count = pick_union(masks, hsv, img_area, 170)       # strict
        method = "auto_strict"
        if count == 0:
            union, count = pick_union(masks, hsv, img_area, 155)   # relaxed
            method = "auto_relaxed"
        if extra_boxes is not None:
            predictor.set_image(img_rgb)
            for b in extra_boxes:
                union = union | box_mask(b, h, w)
            method = method + "_plus_box"

    final_mask = fill_holes(union)
    cv2.imwrite(FINAL_MASK + "/" + stem + ".png", final_mask.astype(np.uint8) * 255)
    overlay = img_rgb.copy()
    overlay[final_mask] = (0.5 * overlay[final_mask] + 0.5 * np.array([0, 255, 0])).astype(np.uint8)
    cv2.imwrite(FINAL_OVER + "/" + stem + ".jpg", cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    blobs = cv2.connectedComponents(final_mask.astype(np.uint8))[0] - 1
    rows.append({"image": stem, "roboflow_folder": path.split("egg_dataset/")[1].split("/")[0],
                 "method": method, "blobs": blobs, "egg_area_pct": round(100 * final_mask.mean(), 2)})

with open(PROJECT + "/results/reference_log_final.csv", "w", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    wr.writeheader()
    wr.writerows(rows)

# ---- Summary (use these numbers in the report) ----
print("Methods used:", dict(Counter(r["method"] for r in rows)))
n_box = 0
for r in rows:
    if "box" in r["method"]:
        n_box = n_box + 1
print("Images that used a manual box:", n_box, "of", len(rows), "=", round(100 * n_box / len(rows), 1), "%")
print("Blobs per image:", dict(sorted(Counter(r["blobs"] for r in rows).items())))
zero = [r["image"] for r in rows if r["blobs"] == 0]
print("Images with ZERO eggs:", len(zero), zero)

# ---- Grid of every image that used a box ----
box_rows = [r for r in rows if "box" in r["method"]]
print("\nImages shown in the grid:")
rws = (len(box_rows) + 3) // 4
plt.figure(figsize=(20, 5 * rws))
for i in range(len(box_rows)):
    r = box_rows[i]
    print("  F" + str(i + 1), r["image"][:28], "|", r["method"], "| blobs", r["blobs"])
    ov = cv2.cvtColor(cv2.imread(FINAL_OVER + "/" + r["image"] + ".jpg"), cv2.COLOR_BGR2RGB)
    plt.subplot(rws, 4, i + 1)
    plt.imshow(ov)
    plt.title("F" + str(i + 1) + "  blobs " + str(r["blobs"]), fontsize=12)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ===== CELL 10: Leakage-aware 80:20 split =====
import os, glob, csv
import numpy as np
import cv2
import matplotlib.pyplot as plt
from collections import Counter

PROJECT = "/content/drive/MyDrive/egg_project"
LOG_PATH = PROJECT + "/results/reference_log_final.csv"
MANIFEST = PROJECT + "/experiments/split_manifest.csv"
os.makedirs(PROJECT + "/experiments", exist_ok=True)

SEED = 42
HASH_THRESHOLD = 10     # max differing bits (out of 64) to call two images near-duplicates
EVAL_FRACTION = 0.20

# 1. Load the image list and the reference-mask info
ref_rows = list(csv.DictReader(open(LOG_PATH)))
stem_to_path = {}
for p in glob.glob("/content/egg_dataset/**/*.jpg", recursive=True):
    stem_to_path[os.path.splitext(os.path.basename(p))[0]] = p
stems = sorted([r["image"] for r in ref_rows])
ref_info = {}
for r in ref_rows:
    ref_info[r["image"]] = r
N = len(stems)
print("Images:", N)

# 2. dHash fingerprint for each image
def dhash(path):
    gray = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2GRAY)
    small = cv2.resize(gray, (9, 8))
    bits = small[:, 1:] > small[:, :-1]      # 8 x 8 = 64 true/false values
    return bits.flatten()

hashes = []
for s in stems:
    hashes.append(dhash(stem_to_path[s]))

# 3. Pairwise distances
dist = np.zeros((N, N), dtype=int)
for i in range(N):
    for j in range(N):
        dist[i, j] = int(np.sum(hashes[i] != hashes[j]))

# nearest neighbour distance for each image (ignore itself)
nearest = []
for i in range(N):
    row = dist[i].copy()
    row[i] = 999
    nearest.append(int(row.min()))
print("Smallest distance to another image: min", min(nearest), "| median", int(np.median(nearest)), "| max", max(nearest))
print("Pairs at or below threshold", HASH_THRESHOLD, ":",
      int(((dist <= HASH_THRESHOLD).sum() - N) // 2))

# 4. Group near-duplicates (union-find)
parent = list(range(N))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for i in range(N):
    for j in range(i + 1, N):
        if dist[i, j] <= HASH_THRESHOLD:
            parent[find(i)] = find(j)

groups = {}
for i in range(N):
    root = find(i)
    if root not in groups:
        groups[root] = []
    groups[root].append(i)
group_list = list(groups.values())
sizes = sorted([len(g) for g in group_list], reverse=True)
print("Groups:", len(group_list), "| biggest group sizes:", sizes[:8])

# 5. Split whole groups with a fixed seed
rng = np.random.RandomState(SEED)
order = rng.permutation(len(group_list))
target_eval = int(round(EVAL_FRACTION * N))
eval_idx = []
for k in order:
    g = group_list[k]
    if len(eval_idx) + len(g) <= target_eval:
        eval_idx = eval_idx + g
eval_set = set(eval_idx)

split_of = {}
group_id_of = {}
for gnum in range(len(group_list)):
    for i in group_list[gnum]:
        group_id_of[i] = gnum
for i in range(N):
    split_of[i] = "eval" if i in eval_set else "dev"

dev = [i for i in range(N) if split_of[i] == "dev"]
ev = [i for i in range(N) if split_of[i] == "eval"]
print("\nDevelopment:", len(dev), "(", round(100 * len(dev) / N, 1), "%)  |  Evaluation:", len(ev), "(", round(100 * len(ev) / N, 1), "%)")

# 6. Leakage checks
dev_groups = set(group_id_of[i] for i in dev)
eval_groups = set(group_id_of[i] for i in ev)
print("Shared groups between dev and eval:", len(dev_groups & eval_groups), "(must be 0)")
print("Shared images between dev and eval:", len(set(dev) & set(ev)), "(must be 0)")
cross = []
for i in dev:
    for j in ev:
        cross.append(dist[i, j])
print("Smallest hash distance across the split:", min(cross), "(should be above", HASH_THRESHOLD, ")")

# 7. Balance check: how are manual-box masks and blob counts spread?
print("\nMask method per split:")
for name, idx in [("dev", dev), ("eval", ev)]:
    print("  ", name, dict(Counter(ref_info[stems[i]]["method"] for i in idx)))
print("Eggs per image (blobs) per split:")
for name, idx in [("dev", dev), ("eval", ev)]:
    print("  ", name, dict(sorted(Counter(ref_info[stems[i]]["blobs"] for i in idx).items())))

# 8. Save the manifest
with open(MANIFEST, "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["image", "group_id", "split", "roboflow_folder", "mask_method", "blobs", "seed", "hash_threshold"])
    for i in range(N):
        s = stems[i]
        wr.writerow([s, group_id_of[i], split_of[i], ref_info[s]["roboflow_folder"],
                     ref_info[s]["method"], ref_info[s]["blobs"], SEED, HASH_THRESHOLD])
print("\nManifest saved:", MANIFEST)

# 9. Look at the biggest groups to check they are real duplicates
big = [g for g in group_list if len(g) > 1]
big = sorted(big, key=len, reverse=True)[:3]
if len(big) == 0:
    print("\nNo near-duplicate groups found at this threshold.")
else:
    plt.figure(figsize=(16, 4 * len(big)))
    for r in range(len(big)):
        for c in range(min(4, len(big[r]))):
            plt.subplot(len(big), 4, r * 4 + c + 1)
            plt.imshow(cv2.cvtColor(cv2.imread(stem_to_path[stems[big[r][c]]]), cv2.COLOR_BGR2RGB))
            plt.title("group of " + str(len(big[r])) + " | " + split_of[big[r][c]], fontsize=9)
            plt.axis("off")
    plt.tight_layout()
    plt.show()

# 10. Look at the closest dev/eval pair
best = (999, 0, 0)
for i in dev:
    for j in ev:
        if dist[i, j] < best[0]:
            best = (dist[i, j], i, j)
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(cv2.imread(stem_to_path[stems[best[1]]]), cv2.COLOR_BGR2RGB)); plt.title("dev"); plt.axis("off")
plt.subplot(1, 2, 2); plt.imshow(cv2.cvtColor(cv2.imread(stem_to_path[stems[best[2]]]), cv2.COLOR_BGR2RGB)); plt.title("eval  (distance " + str(best[0]) + ")"); plt.axis("off")
plt.show()

In [ ]:
# ===== CELL 11: Experiment framework + E1 baseline (development set only) =====
import os, csv, glob
import numpy as np
import cv2
import matplotlib.pyplot as plt

PROJECT = "/content/drive/MyDrive/egg_project"
MANIFEST = PROJECT + "/experiments/split_manifest.csv"
LOG_CSV = PROJECT + "/experiments/experiment_log.csv"
REF_DIR = PROJECT + "/results/reference_masks"

# ---------- 1. Load DEVELOPMENT images only ----------
stem_to_path = {}
for p in glob.glob("/content/egg_dataset/**/*.jpg", recursive=True):
    stem_to_path[os.path.splitext(os.path.basename(p))[0]] = p

dev_data = []   # list of (stem, image_bgr, reference_mask_bool)
for r in csv.DictReader(open(MANIFEST)):
    if r["split"] == "dev":
        img = cv2.imread(stem_to_path[r["image"]])
        ref = cv2.imread(REF_DIR + "/" + r["image"] + ".png", 0) > 0
        dev_data.append((r["image"], img, ref))
print("Development images loaded:", len(dev_data))

# ---------- 2. Metrics ----------
def compute_metrics(pred, ref):
    tp = int(np.logical_and(pred, ref).sum())
    fp = int(np.logical_and(pred, ~ref).sum())
    fn = int(np.logical_and(~pred, ref).sum())
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 1.0
    dice = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 1.0
    if tp + fp > 0:
        precision = tp / (tp + fp)
    else:
        precision = 1.0 if fn == 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 1.0
    return {"tp": tp, "fp": fp, "fn": fn, "iou": iou, "dice": dice,
            "precision": precision, "recall": recall}

# ---------- 3. Experiment log helpers ----------
LOG_COLS = ["trial_id", "hypothesis", "method_change", "parameters", "motivation",
            "dev_macro_iou", "dev_macro_dice", "dev_macro_precision", "dev_macro_recall",
            "dev_micro_iou", "observations", "decision", "next_step"]

def read_log():
    if os.path.exists(LOG_CSV):
        return list(csv.DictReader(open(LOG_CSV)))
    return []

def write_log(rows):
    with open(LOG_CSV, "w", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=LOG_COLS)
        wr.writeheader()
        wr.writerows(rows)

def log_row(row):
    rows = [r for r in read_log() if r["trial_id"] != row["trial_id"]]   # replace same id
    rows.append(row)
    write_log(rows)

def record_decision(trial_id, observations, decision, next_step):
    rows = read_log()
    for r in rows:
        if r["trial_id"] == trial_id:
            r["observations"] = observations
            r["decision"] = decision
            r["next_step"] = next_step
    write_log(rows)
    print("Recorded decision for", trial_id)

# ---------- 4. Run one experiment on the development set ----------
def run_experiment(trial_id, hypothesis, change, params, motivation, segment_fn, show=True):
    out_dir = PROJECT + "/results/classical_masks/" + trial_id
    os.makedirs(out_dir, exist_ok=True)
    results = []
    for stem, img, ref in dev_data:
        pred = segment_fn(img, params)
        cv2.imwrite(out_dir + "/" + stem + ".png", pred.astype(np.uint8) * 255)
        m = compute_metrics(pred, ref)
        m["stem"] = stem
        m["pred"] = pred
        results.append(m)

    macro = {}
    for k in ["iou", "dice", "precision", "recall"]:
        macro[k] = float(np.mean([r[k] for r in results]))
    TP = sum(r["tp"] for r in results)
    FP = sum(r["fp"] for r in results)
    FN = sum(r["fn"] for r in results)
    micro_iou = TP / (TP + FP + FN)
    micro_prec = TP / (TP + FP) if (TP + FP) > 0 else 0
    micro_rec = TP / (TP + FN) if (TP + FN) > 0 else 0

    print("=====", trial_id, "(development set,", len(results), "images) =====")
    print("MACRO  IoU %.3f | Dice %.3f | Precision %.3f | Recall %.3f" %
          (macro["iou"], macro["dice"], macro["precision"], macro["recall"]))
    print("MICRO  IoU %.3f | Precision %.3f | Recall %.3f" % (micro_iou, micro_prec, micro_rec))

    log_row({"trial_id": trial_id, "hypothesis": hypothesis, "method_change": change,
             "parameters": str(params), "motivation": motivation,
             "dev_macro_iou": round(macro["iou"], 4), "dev_macro_dice": round(macro["dice"], 4),
             "dev_macro_precision": round(macro["precision"], 4),
             "dev_macro_recall": round(macro["recall"], 4),
             "dev_micro_iou": round(micro_iou, 4),
             "observations": "pending", "decision": "pending", "next_step": "pending"})

    if show:
        show_cases(results)
    return results

# ---------- 5. Visualise: original | reference | prediction | errors ----------
def show_cases(results):
    rng = np.random.RandomState(0)
    random_idx = list(rng.choice(len(results), 3, replace=False))
    order = sorted(range(len(results)), key=lambda i: results[i]["iou"])
    worst_idx = order[:2]
    chosen = random_idx + worst_idx
    labels = ["random", "random", "random", "WORST", "WORST"]
    plt.figure(figsize=(16, 4 * len(chosen)))
    for n in range(len(chosen)):
        i = chosen[n]
        stem, img, ref = dev_data[i]
        pred = results[i]["pred"]
        err = np.zeros((ref.shape[0], ref.shape[1], 3), dtype=np.uint8)
        err[np.logical_and(pred, ref)] = [0, 200, 0]     # correct = green
        err[np.logical_and(pred, ~ref)] = [255, 0, 0]    # false positive = red
        err[np.logical_and(~pred, ref)] = [0, 80, 255]   # missed = blue
        panels = [cv2.cvtColor(img, cv2.COLOR_BGR2RGB), ref, pred, err]
        titles = ["original", "reference (SAM)", "prediction", "IoU %.2f  green ok / red extra / blue missed" % results[i]["iou"]]
        for c in range(4):
            plt.subplot(len(chosen), 4, n * 4 + c + 1)
            plt.imshow(panels[c], cmap="gray")
            plt.title(labels[n] + ": " + titles[c] if c == 0 else titles[c], fontsize=9)
            plt.axis("off")
    plt.tight_layout()
    plt.show()
    print("Worst 5 images by IoU:")
    for i in order[:5]:
        print("   ", results[i]["stem"][:30], "IoU %.2f  precision %.2f  recall %.2f" %
              (results[i]["iou"], results[i]["precision"], results[i]["recall"]))

# ---------- 6. E1: grayscale + Otsu ----------
def segment_e1(img, params):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary > 0

e1_results = run_experiment(
    trial_id="E1",
    hypothesis="Eggs are white, so a bright-pixel threshold on grayscale separates them from the background.",
    change="Baseline: grayscale + global Otsu threshold, nothing else",
    params={"method": "Otsu", "blur": "none"},
    motivation="Simplest possible baseline to measure everything else against.",
    segment_fn=segment_e1)

In [ ]:
# ===== CELL 12: record E1 + run E2 (HSV colour threshold) =====
import numpy as np
import cv2

if "run_experiment" not in globals():
    raise RuntimeError("Run Cell 11 first in this session.")

# ---- 1. Record the E1 decision in the log ----
record_decision(
    "E1",
    "Macro IoU 0.345, precision 0.38, recall 0.87. Finds eggs but also marks bright backgrounds and a finger; images 20 and 44 almost fully missed (egg not bright).",
    "keep as baseline, not final",
    "E2: HSV threshold (low saturation + high brightness) to drop coloured things like fingers")

# ---- 2. E2 segmentation function ----
def segment_e2(img, params):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    sat = hsv[:, :, 1]
    val = hsv[:, :, 2]
    return (sat <= params["sat_max"]) & (val >= params["val_min"])

# ---- 3. Small sweep on the DEVELOPMENT set only ----
print("Sweep on development set (macro IoU):")
print("  sat_max  val_min  macro_IoU")
best_score = -1
best_params = None
for sat_max in [40, 70, 100]:
    for val_min in [140, 170, 200]:
        scores = []
        for stem, img, ref in dev_data:
            pred = segment_e2(img, {"sat_max": sat_max, "val_min": val_min})
            scores.append(compute_metrics(pred, ref)["iou"])
        score = float(np.mean(scores))
        print("  ", sat_max, "      ", val_min, "     ", round(score, 3))
        if score > best_score:
            best_score = score
            best_params = {"sat_max": sat_max, "val_min": val_min}
print("Best:", best_params, "->", round(best_score, 3))

# ---- 4. Run E2 with the best parameters (logs it, shows pictures) ----
e2_results = run_experiment(
    trial_id="E2",
    hypothesis="Eggs are white: low saturation and high brightness. A colour rule should remove coloured background and fingers that plain brightness keeps.",
    change="Replace grayscale Otsu with HSV rule: saturation <= sat_max AND value >= val_min",
    params=best_params,
    motivation="E1 had precision 0.38: it marked fingers and coloured background. Saturation separates white from coloured things.",
    segment_fn=segment_e2)

In [ ]:
# ===== CELL 13: record E2 + run E3 (morphology on top of E2) =====
import numpy as np
import cv2

if "run_experiment" not in globals():
    raise RuntimeError("Run Cell 11 first in this session.")

# ---- 1. Record the E2 decision ----
record_decision(
    "E2",
    "Macro IoU 0.424 (up from 0.345), precision 0.55, recall 0.70. Best sat_max=40, val_min=170. Dim or shadowed eggs (images 39, 20, 119, 44) almost fully missed; white panel behind egg (125) and a finger still marked.",
    "keep, current best",
    "E3: add morphological opening and closing to remove specks and fill gaps")

E2_PARAMS = {"sat_max": 40, "val_min": 170}

# ---- 2. E3 segmentation function ----
def clean_mask(mask_bool, open_k, close_k):
    m = mask_bool.astype(np.uint8) * 255
    if open_k > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_k, open_k))
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, kernel)
    if close_k > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_k, close_k))
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel)
    return m > 0

def segment_e3(img, params):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = (hsv[:, :, 1] <= params["sat_max"]) & (hsv[:, :, 2] >= params["val_min"])
    return clean_mask(mask, params["open_k"], params["close_k"])

# ---- 3. Sweep on the DEVELOPMENT set only ----
print("Sweep on development set (macro IoU):")
print("  open_k  close_k  macro_IoU")
best_score = -1
best_params = None
for open_k in [0, 5, 9]:
    for close_k in [0, 9, 17]:
        params = {"sat_max": 40, "val_min": 170, "open_k": open_k, "close_k": close_k}
        scores = []
        for stem, img, ref in dev_data:
            pred = segment_e3(img, params)
            scores.append(compute_metrics(pred, ref)["iou"])
        score = float(np.mean(scores))
        print("  ", open_k, "     ", close_k, "     ", round(score, 3))
        if score > best_score:
            best_score = score
            best_params = params
print("Best:", best_params, "->", round(best_score, 3))

# ---- 4. Run E3 with the best parameters ----
e3_results = run_experiment(
    trial_id="E3",
    hypothesis="Opening removes specks and thin bridges (like a finger link); closing fills gaps from dirt and reflections inside eggs.",
    change="Add morphological opening then closing (elliptical kernel) after the E2 colour rule",
    params=best_params,
    motivation="E2 masks are ragged, with holes inside eggs and small false specks in the background.",
    segment_fn=segment_e3)

In [ ]:
# ===== CELL 14: record E3 + run E4 (blob filtering on top of E3) =====
import numpy as np
import cv2

if "run_experiment" not in globals() or "segment_e3" not in globals():
    raise RuntimeError("Run Cells 11, 12 and 13 first in this session.")

# ---- 1. Record the E3 decision ----
record_decision(
    "E3",
    "Macro IoU 0.447 (up from 0.424), precision 0.586, recall 0.690. Opening kernel 9 best (largest value tried, so larger not explored); closing hurt in every setting and was dropped. Dim-egg images and white panel image still fail.",
    "keep, current best",
    "E4: filter blobs by size, border contact and elongation to drop panels, fingers and specks")

E3_PARAMS = {"sat_max": 40, "val_min": 170, "open_k": 9, "close_k": 0}

# ---- 2. Blob filter ----
def filter_blobs(mask_bool, min_area_frac, max_edge, max_aspect):
    h, w = mask_bool.shape
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask_bool.astype(np.uint8))
    out = np.zeros((h, w), dtype=bool)
    for k in range(1, n):
        area = stats[k, cv2.CC_STAT_AREA]
        if area < min_area_frac * h * w:
            continue                                   # too small: speck
        comp = labels == k
        border = comp[0, :].sum() + comp[-1, :].sum() + comp[:, 0].sum() + comp[:, -1].sum()
        edge = border / (2 * (h + w))
        if edge > max_edge:
            continue                                   # wraps around the border: panel/wall
        contours, _ = cv2.findContours(comp.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        big = max(contours, key=cv2.contourArea)
        (cx, cy), (rw, rh), angle = cv2.minAreaRect(big)
        short_side = min(rw, rh)
        aspect = max(rw, rh) / short_side if short_side > 0 else 99
        if aspect > max_aspect:
            continue                                   # too long and thin: finger/strip
        out[comp] = True
    return out

def segment_e4(img, params):
    base = segment_e3(img, E3_PARAMS)
    return filter_blobs(base, params["min_area_frac"], params["max_edge"], params["max_aspect"])

# ---- 3. Sweep on the DEVELOPMENT set only (E3 masks computed once) ----
base_masks = []
for stem, img, ref in dev_data:
    base_masks.append(segment_e3(img, E3_PARAMS))

print("Sweep on development set (macro IoU):")
print("  min_area  max_edge  macro_IoU")
best_score = -1
best_params = None
for min_area in [0.01, 0.02, 0.04]:
    for max_edge in [0.15, 0.30, 1.0]:
        scores = []
        for i in range(len(dev_data)):
            pred = filter_blobs(base_masks[i], min_area, max_edge, 2.5)
            scores.append(compute_metrics(pred, dev_data[i][2])["iou"])
        score = float(np.mean(scores))
        print("  ", min_area, "     ", max_edge, "     ", round(score, 3))
        if score > best_score:
            best_score = score
            best_params = {"min_area_frac": min_area, "max_edge": max_edge, "max_aspect": 2.5}
print("Best:", best_params, "->", round(best_score, 3))

# ---- 4. Run E4 with the best parameters ----
e4_results = run_experiment(
    trial_id="E4",
    hypothesis="Backgrounds and fingers form blobs that are large and wrap the image border, or are long and thin; eggs are compact blobs inside the picture.",
    change="Add connected-component blob filter (min size, border contact, aspect ratio) after E3",
    params=best_params,
    motivation="E3 still marks a white panel (image 125) and a finger; colour alone cannot tell them from eggs, shape and position can.",
    segment_fn=segment_e4)


In [ ]:
# ===== CELL 15: record E4, diagnose images 149/150, run E5 (watershed) =====
import numpy as np
import cv2

if "run_experiment" not in globals() or "filter_blobs" not in globals():
    raise RuntimeError("Run Cells 11, 13 and 14 first in this session.")

# ---- 1. Record the E4 decision ----
record_decision(
    "E4",
    "Macro IoU 0.492 (up from 0.447), precision 0.688, recall 0.660. Best min_area 2% of image; border-contact filter hurt in every setting (close-up eggs touch the border) so max_edge=1.0 (off). New zero-IoU images 149 and 150 suggest the filter removed their eggs.",
    "keep, current best",
    "E5: watershed to split touching eggs before the blob filter")

E4_PARAMS = {"min_area_frac": 0.02, "max_edge": 1.0, "max_aspect": 2.5}

# ---- 2. Diagnose 149 and 150: what do the E3 blobs look like? ----
def describe_blobs(mask_bool):
    h, w = mask_bool.shape
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask_bool.astype(np.uint8))
    if n == 1:
        print("      no blobs at all in the E3 mask")
    for k in range(1, n):
        comp = labels == k
        contours, _ = cv2.findContours(comp.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        big = max(contours, key=cv2.contourArea)
        (cx, cy), (rw, rh), angle = cv2.minAreaRect(big)
        aspect = max(rw, rh) / min(rw, rh) if min(rw, rh) > 0 else 99
        area_frac = stats[k, cv2.CC_STAT_AREA] / (h * w)
        verdict = "keep"
        if area_frac < 0.02:
            verdict = "REMOVED: too small"
        elif aspect > 2.5:
            verdict = "REMOVED: too long/thin"
        print("      blob", k, "| area percent:", round(100 * area_frac, 1),
              "| aspect:", round(aspect, 2), "|", verdict)

print("Diagnosis of the two new zero-score images (E3 blobs and what E4 does):")
for prefix in ["not_damaged_149_jpg", "not_damaged_150_jpg"]:
    for i in range(len(dev_data)):
        if dev_data[i][0].startswith(prefix):
            n_ref = cv2.connectedComponents(dev_data[i][2].astype(np.uint8))[0] - 1
            print("  ", prefix, "| reference eggs (blobs):", n_ref)
            describe_blobs(segment_e3(dev_data[i][1], E3_PARAMS))

# ---- 3. Watershed splitting ----
def split_touching(mask_bool, img_bgr, peak_frac):
    h, w = mask_bool.shape
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask_bool.astype(np.uint8))
    markers = np.zeros((h, w), dtype=np.int32)
    next_id = 2                                   # label 1 is kept for the background
    for k in range(1, n):
        comp = (labels == k).astype(np.uint8)
        dist = cv2.distanceTransform(comp, cv2.DIST_L2, 5)
        if dist.max() == 0:
            continue
        core = (dist >= peak_frac * dist.max()).astype(np.uint8)    # area near the peak(s)
        m, core_labels = cv2.connectedComponents(core)
        for c in range(1, m):
            markers[core_labels == c] = next_id
            next_id = next_id + 1
    sure_bg = cv2.dilate(mask_bool.astype(np.uint8), np.ones((3, 3), np.uint8), iterations=2) == 0
    markers[sure_bg] = 1
    ws = cv2.watershed(img_bgr.copy(), markers)
    boundary = (ws == -1).astype(np.uint8)
    boundary = cv2.dilate(boundary, np.ones((3, 3), np.uint8)) > 0   # thicken the cut line
    return (ws > 1) & mask_bool & ~boundary

def segment_e5(img, params):
    base = segment_e3(img, E3_PARAMS)
    split = split_touching(base, img, params["peak_frac"])
    return filter_blobs(split, E4_PARAMS["min_area_frac"], E4_PARAMS["max_edge"], E4_PARAMS["max_aspect"])

# ---- 4. Sweep on the DEVELOPMENT set only ----
print("\nSweep on development set (macro IoU). Higher peak_frac = more splitting.")
print("  peak_frac  macro_IoU")
best_score = -1
best_params = None
for peak in [0.5, 0.6, 0.7, 0.8]:
    scores = []
    for stem, img, ref in dev_data:
        pred = segment_e5(img, {"peak_frac": peak})
        scores.append(compute_metrics(pred, ref)["iou"])
    score = float(np.mean(scores))
    print("  ", peak, "      ", round(score, 3))
    if score > best_score:
        best_score = score
        best_params = {"peak_frac": peak}
print("Best:", best_params, "->", round(best_score, 3), "(E4 was 0.492)")

# ---- 5. Run E5 with the best setting ----
e5_results = run_experiment(
    trial_id="E5",
    hypothesis="Touching eggs form one merged blob that fails the shape filter; splitting them with watershed first lets each egg pass.",
    change="Insert distance-transform watershed (per-blob peaks) between E3 cleaning and the E4 blob filter",
    params=best_params,
    motivation="E4 removed some eggs (images 149, 150) likely because merged eggs became one long blob.",
    segment_fn=segment_e5)

In [ ]:
# ===== CELL 16: record E5 (discarded) + run E6 (looser aspect limit) =====
import numpy as np
import cv2

if "run_experiment" not in globals() or "filter_blobs" not in globals() or "segment_e3" not in globals():
    raise RuntimeError("Run Cells 11, 13 and 14 first in this session.")

# ---- 1. Record the E5 decision ----
record_decision(
    "E5",
    "Macro IoU 0.442 (worse than E4 0.492), recall dropped to 0.543. Best peak_frac was 0.5, the setting that splits least, suggesting watershed over-splits single eggs and the cut lines remove pixels (not verified).",
    "discard",
    "E6: images 149/150 lost eggs because blob aspect ratio 2.7-3.6 failed the 2.5 limit; tune max_aspect")

E3_PARAMS = {"sat_max": 40, "val_min": 170, "open_k": 9, "close_k": 0}

# ---- 2. E6 segmentation function ----
def segment_e6(img, params):
    base = segment_e3(img, E3_PARAMS)
    return filter_blobs(base, 0.02, 1.0, params["max_aspect"])

# ---- 3. Sweep on the DEVELOPMENT set only (E3 masks computed once) ----
base_masks = []
for stem, img, ref in dev_data:
    base_masks.append(segment_e3(img, E3_PARAMS))

print("Sweep on development set (macro IoU). 99 means no aspect limit.")
print("  max_aspect  macro_IoU")
best_score = -1
best_params = None
for max_aspect in [2.5, 3.0, 3.5, 4.0, 99]:
    scores = []
    for i in range(len(dev_data)):
        pred = filter_blobs(base_masks[i], 0.02, 1.0, max_aspect)
        scores.append(compute_metrics(pred, dev_data[i][2])["iou"])
    score = float(np.mean(scores))
    print("  ", max_aspect, "      ", round(score, 3))
    if score > best_score:
        best_score = score
        best_params = {"max_aspect": max_aspect}
print("Best:", best_params, "->", round(best_score, 3), "(E4 was 0.492)")

# ---- 4. Run E6 with the best setting ----
e6_results = run_experiment(
    trial_id="E6",
    hypothesis="Touching eggs form long blobs (aspect 2.7-3.6); the 2.5 limit removed them, so a looser limit recovers them.",
    change="Keep E3 + E4 filter (min area 2%, no border rule); only raise max_aspect",
    params={"min_area_frac": 0.02, "max_edge": 1.0, "max_aspect": best_params["max_aspect"]},
    motivation="Diagnosis of images 149 and 150 showed egg blobs removed only by the aspect-ratio check.",
    segment_fn=segment_e6)


In [ ]:
# ===== CELL 17: FINAL evaluation on the held-out evaluation set (run once) =====
import os, csv
import numpy as np
import cv2
import matplotlib.pyplot as plt

needed = ["segment_e6", "compute_metrics", "filter_blobs", "segment_e3", "E3_PARAMS", "stem_to_path", "dev_data"]
for name in needed:
    if name not in globals():
        raise RuntimeError("Run Cells 11, 13, 14 and 16 first in this session. Missing: " + name)

# ---- 1. Record E6 and freeze parameters ----
record_decision(
    "E6",
    "Macro IoU 0.496 (E4 was 0.492). max_aspect 3.0 recovered touching-egg blobs (images 149/150). Worst images are dim or shadowed eggs; blue in error maps is mostly egg pixels below the brightness limit.",
    "keep as FINAL algorithm",
    "Freeze parameters; run one final evaluation on the held-out evaluation set")

FINAL_PARAMS = {"max_aspect": 3.0}     # other parameters are fixed inside segment_e3 / segment_e6
PROJECT = "/content/drive/MyDrive/egg_project"
PLOT_DIR = PROJECT + "/results/plots"
PRED_DIR = PROJECT + "/results/classical_masks/FINAL_eval"
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

# ---- 2. Load the EVALUATION images only ----
eval_data = []     # (stem, image, reference mask, mask_method)
for r in csv.DictReader(open(MANIFEST)):
    if r["split"] == "eval":
        img = cv2.imread(stem_to_path[r["image"]])
        ref = cv2.imread(REF_DIR + "/" + r["image"] + ".png", 0) > 0
        eval_data.append((r["image"], img, ref, r["mask_method"]))
print("Evaluation images:", len(eval_data))

# ---- 3. Run the frozen algorithm and score it ----
def count_blobs(mask_bool):
    return cv2.connectedComponents(mask_bool.astype(np.uint8))[0] - 1

rows = []
preds = []
for stem, img, ref, method in eval_data:
    pred = segment_e6(img, FINAL_PARAMS)
    preds.append(pred)
    cv2.imwrite(PRED_DIR + "/" + stem + ".png", pred.astype(np.uint8) * 255)
    m = compute_metrics(pred, ref)
    m["image"] = stem
    m["mask_method"] = method
    m["ref_eggs"] = count_blobs(ref)
    m["pred_eggs"] = count_blobs(pred)
    rows.append(m)

cols = ["image", "mask_method", "ref_eggs", "pred_eggs", "tp", "fp", "fn", "iou", "dice", "precision", "recall"]
with open(PROJECT + "/results/eval_metrics_per_image.csv", "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(cols)
    for r in rows:
        wr.writerow([r[c] if not isinstance(r[c], float) else round(r[c], 4) for c in cols])

# ---- 4. Aggregate scores (macro and micro), eval vs development ----
def summarise(metric_rows):
    out = {}
    for k in ["iou", "dice", "precision", "recall"]:
        vals = [r[k] for r in metric_rows]
        out["macro_" + k] = float(np.mean(vals))
        out["median_" + k] = float(np.median(vals))
        out["std_" + k] = float(np.std(vals))
    TP = sum(r["tp"] for r in metric_rows)
    FP = sum(r["fp"] for r in metric_rows)
    FN = sum(r["fn"] for r in metric_rows)
    out["micro_iou"] = TP / (TP + FP + FN)
    out["micro_dice"] = 2 * TP / (2 * TP + FP + FN)
    out["micro_precision"] = TP / (TP + FP) if TP + FP > 0 else 0
    out["micro_recall"] = TP / (TP + FN) if TP + FN > 0 else 0
    return out

dev_rows = []
for stem, img, ref in dev_data:
    dev_rows.append(compute_metrics(segment_e6(img, FINAL_PARAMS), ref))
ev_sum = summarise(rows)
dv_sum = summarise(dev_rows)

print("\n===== FINAL RESULTS (E6) =====")
print("                      dev(88)   eval(22)")
for k in ["macro_iou", "macro_dice", "macro_precision", "macro_recall",
          "median_iou", "micro_iou", "micro_dice", "micro_precision", "micro_recall"]:
    print("  %-18s  %.3f     %.3f" % (k, dv_sum[k], ev_sum[k]))
print("  eval IoU spread (std):", round(ev_sum["std_iou"], 3))

exact = sum(1 for r in rows if r["ref_eggs"] == r["pred_eggs"])
print("\nEgg counting: predicted blob count equals reference in", exact, "of", len(rows), "images")
print("Mean absolute count error: %.2f eggs" % np.mean([abs(r["ref_eggs"] - r["pred_eggs"]) for r in rows]))

with open(PROJECT + "/results/eval_metrics_summary.csv", "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["metric", "dev", "eval"])
    for k in ev_sum:
        wr.writerow([k, round(dv_sum[k], 4), round(ev_sum[k], 4)])

# ---- 5. Charts ----
auto_rows = [r for r in rows if "box" not in r["mask_method"]]
box_rows = [r for r in rows if "box" in r["mask_method"]]
fig = plt.figure(figsize=(16, 11))

plt.subplot(2, 2, 1)
plt.hist([r["iou"] for r in rows], bins=10, range=(0, 1), color="steelblue", edgecolor="black")
plt.xlabel("IoU per image"); plt.ylabel("number of images"); plt.title("IoU distribution (evaluation set)")

plt.subplot(2, 2, 2)
plt.scatter([r["precision"] for r in auto_rows], [r["recall"] for r in auto_rows], label="auto mask (" + str(len(auto_rows)) + ")", s=60)
plt.scatter([r["precision"] for r in box_rows], [r["recall"] for r in box_rows], label="manual-box mask (" + str(len(box_rows)) + ")", s=60, marker="s")
plt.xlabel("precision"); plt.ylabel("recall"); plt.xlim(-0.05, 1.05); plt.ylim(-0.05, 1.05)
plt.legend(); plt.title("Precision vs recall per image")

plt.subplot(2, 2, 3)
labels, values = [], []
for name, group in [("auto mask", auto_rows), ("manual-box mask", box_rows)]:
    if len(group) > 0:
        labels.append(name + "\n(n=" + str(len(group)) + ")")
        values.append(np.mean([r["iou"] for r in group]))
plt.bar(labels, values, color=["steelblue", "orange"][:len(values)])
plt.ylabel("mean IoU"); plt.ylim(0, 1); plt.title("IoU by how the reference was made")

plt.subplot(2, 2, 4)
labels, values = [], []
for n_eggs in [1, 2, 3]:
    group = [r for r in rows if r["ref_eggs"] == n_eggs]
    if len(group) > 0:
        labels.append(str(n_eggs) + " egg(s)\n(n=" + str(len(group)) + ")")
        values.append(np.mean([r["iou"] for r in group]))
plt.bar(labels, values, color="seagreen")
plt.ylabel("mean IoU"); plt.ylim(0, 1); plt.title("IoU by number of eggs in the reference")

plt.tight_layout()
plt.savefig(PLOT_DIR + "/eval_quantitative.png", dpi=120)
plt.show()

# ---- 6. Failure gallery: 6 worst images ----
order = sorted(range(len(rows)), key=lambda i: rows[i]["iou"])
worst = order[:6]
plt.figure(figsize=(16, 4 * len(worst)))
print("\nWorst 6 evaluation images (F1 ... F6, top to bottom):")
for n in range(len(worst)):
    i = worst[n]
    stem, img, ref, method = eval_data[i]
    pred = preds[i]
    err = np.zeros((ref.shape[0], ref.shape[1], 3), dtype=np.uint8)
    err[np.logical_and(pred, ref)] = [0, 200, 0]
    err[np.logical_and(pred, ~ref)] = [255, 0, 0]
    err[np.logical_and(~pred, ref)] = [0, 80, 255]
    print("  F" + str(n + 1), stem[:28], "| IoU %.2f precision %.2f recall %.2f | ref eggs %d pred eggs %d | %s" %
          (rows[i]["iou"], rows[i]["precision"], rows[i]["recall"], rows[i]["ref_eggs"], rows[i]["pred_eggs"], method))
    panels = [cv2.cvtColor(img, cv2.COLOR_BGR2RGB), ref, pred, err]
    titles = ["F" + str(n + 1) + " original", "reference (SAM)", "prediction", "green ok / red extra / blue missed"]
    for c in range(4):
        plt.subplot(len(worst), 4, n * 4 + c + 1)
        plt.imshow(panels[c], cmap="gray")
        plt.title(titles[c], fontsize=9)
        plt.axis("off")
plt.tight_layout()
plt.savefig(PLOT_DIR + "/eval_failure_cases.png", dpi=110)
plt.show()
print("\nSaved: eval_metrics_per_image.csv, eval_metrics_summary.csv, plots/ in", PROJECT + "/results")

In [ ]:
# ===== CELL 18: Preview E6 segmentation on video frames (10 s to 40 s) =====
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt

for name in ["segment_e6", "segment_e3", "filter_blobs", "E3_PARAMS"]:
    if name not in globals():
        raise RuntimeError("Run Cells 11, 13, 14 and 16 first in this session. Missing: " + name)

VIDEO_PATH = "/content/drive/MyDrive/egg_project/video/input.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
print("FPS:", fps, "| size:", int(cap.get(3)), "x", int(cap.get(4)))

plt.figure(figsize=(20, 12))
times = []
count = 0
for sec in range(10, 42, 2):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(sec * fps))
    ok, frame = cap.read()
    if not ok:
        continue
    t0 = time.time()
    mask = segment_e6(frame, {"max_aspect": 3.0})
    times.append(time.time() - t0)

    # draw outline + number for each blob
    shown = frame.copy()
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8))
    for k in range(1, n):
        comp = (labels == k).astype(np.uint8)
        contours, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(shown, contours, -1, (0, 255, 0), 3)
        cv2.putText(shown, str(k), (int(centroids[k][0]), int(centroids[k][1])),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

    count += 1
    plt.subplot(4, 4, count)
    plt.imshow(cv2.cvtColor(shown, cv2.COLOR_BGR2RGB))
    plt.title(str(sec) + " s  |  blobs: " + str(n - 1), fontsize=11)
    plt.axis("off")
cap.release()
plt.tight_layout()
plt.show()
print("Average time per frame: %.3f s  ->  full 745 frames about %.0f s" % (np.mean(times), np.mean(times) * 745))

In [ ]:
# ===== CELL 19: 4 video frames with a pixel grid =====
import cv2
import matplotlib.pyplot as plt

VIDEO_PATH = "/content/drive/MyDrive/egg_project/video/input.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)

plt.figure(figsize=(22, 13))
count = 0
for sec in [14, 20, 32, 34]:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(sec * fps))
    ok, frame = cap.read()
    if not ok:
        continue
    count += 1
    plt.subplot(2, 2, count)
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.xticks(range(0, 855, 50))
    plt.yticks(range(0, 481, 50))
    plt.grid(True, color="yellow", alpha=0.5)
    plt.title(str(sec) + " s", fontsize=16)
cap.release()
plt.tight_layout()
plt.show()

In [ ]:
# ===== CELL 20: video-adapted thresholds (trial V1) =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

for name in ["clean_mask", "filter_blobs", "compute_metrics", "log_row", "LOG_COLS"]:
    if name not in globals():
        raise RuntimeError("Run Cells 11, 13, 14 and 16 first in this session. Missing: " + name)

# ---- 1. Boxes from Cell 19: [left, top, right, bottom] for each clearly visible egg ----
BOXES = {
    14: [[0, 200, 275, 350]],
    20: [[25, 200, 275, 350]],
    32: [[25, 200, 150, 325]],
    34: [[25, 200, 150, 325], [450, 275, 600, 400]],
}

VIDEO_PATH = "/content/drive/MyDrive/egg_project/video/input.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)

def get_frame(sec):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(sec * fps))
    ok, frame = cap.read()
    return frame

def oval_proxy(h, w, boxes):
    """Draw an oval inside each box: a rough stand-in for the egg shape."""
    m = np.zeros((h, w), dtype=np.uint8)
    for b in boxes:
        centre = (int((b[0] + b[2]) / 2), int((b[1] + b[3]) / 2))
        axes = (int((b[2] - b[0]) / 2), int((b[3] - b[1]) / 2))
        cv2.ellipse(m, centre, axes, 0, 0, 360, 255, -1)
    return m > 0

def segment_video(frame, params):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = (hsv[:, :, 1] <= params["sat_max"]) & (hsv[:, :, 2] >= params["val_min"])
    mask = clean_mask(mask, 9, 0)                      # same opening as E3
    return filter_blobs(mask, 0.02, 1.0, 3.0)           # same blob filter as E6

# ---- 2. Sweep: score against the oval proxy ----
frames = {}
proxies = {}
for sec in BOXES:
    frames[sec] = get_frame(sec)
    proxies[sec] = oval_proxy(frames[sec].shape[0], frames[sec].shape[1], BOXES[sec])

print("Sweep on the 4 boxed frames (mean proxy IoU, then IoU per frame):")
print("  sat_max val_min  mean   per-frame (" + ", ".join(str(s) + "s" for s in BOXES) + ")")
best_score = -1
best_params = None
for sat_max in [40, 70, 100, 130]:
    for val_min in [100, 120, 140, 170]:
        per = []
        for sec in BOXES:
            pred = segment_video(frames[sec], {"sat_max": sat_max, "val_min": val_min})
            per.append(compute_metrics(pred, proxies[sec])["iou"])
        score = float(np.mean(per))
        print("  ", sat_max, "    ", val_min, "   ", round(score, 3), "  ", [round(p, 2) for p in per])
        if score > best_score:
            best_score = score
            best_params = {"sat_max": sat_max, "val_min": val_min}
print("Best:", best_params, "-> proxy IoU", round(best_score, 3))

log_row({"trial_id": "V1", "hypothesis": "Video eggs are darker/warmer than the dataset eggs, so the dataset thresholds miss them.",
         "method_change": "Video only: retune HSV saturation/brightness limits on 4 frames (oval proxy from manual boxes); opening and blob filter unchanged",
         "parameters": str(best_params), "motivation": "Preview showed missed eggs in about half of the sampled frames (18, 20, 32, 34-40 s).",
         "dev_macro_iou": round(best_score, 4), "dev_macro_dice": "", "dev_macro_precision": "", "dev_macro_recall": "",
         "dev_micro_iou": "", "observations": "Proxy IoU on 4 frames only (not comparable to E1-E6 which use SAM masks)",
         "decision": "pending visual check on 16 preview frames", "next_step": "build tracker"})

# ---- 3. Re-draw the 16-frame preview with the new setting ----
plt.figure(figsize=(20, 12))
count = 0
for sec in range(10, 42, 2):
    frame = get_frame(sec)
    mask = segment_video(frame, best_params)
    shown = frame.copy()
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8))
    for k in range(1, n):
        comp = (labels == k).astype(np.uint8)
        contours, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(shown, contours, -1, (0, 255, 0), 3)
        cv2.putText(shown, str(k), (int(centroids[k][0]), int(centroids[k][1])),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
    count += 1
    plt.subplot(4, 4, count)
    plt.imshow(cv2.cvtColor(shown, cv2.COLOR_BGR2RGB))
    plt.title(str(sec) + " s  |  blobs: " + str(n - 1), fontsize=11)
    plt.axis("off")
cap.release()
plt.tight_layout()
plt.show()

In [ ]:
# ===== CELL 21: V2 = colour + tracking zone + max blob size =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

for name in ["clean_mask", "filter_blobs", "compute_metrics", "log_row", "record_decision"]:
    if name not in globals():
        raise RuntimeError("Run Cells 11, 13, 14 and 16 first in this session. Missing: " + name)

# ---- 1. Record V1 as discarded ----
record_decision(
    "V1",
    "Looser colour limits recovered some eggs but also marked shirts and pants; colour alone cannot separate eggs from pale or grey clothing.",
    "discard",
    "V2: restrict to a horizontal tracking zone where eggs were seen and drop oversized blobs")

# ---- 2. Settings (from the 4 boxed frames: eggs lie between y=200 and y=400) ----
ROI_TOP = 160
ROI_BOTTOM = 440
MAX_AREA_FRAC = 0.12      # eggs cover about 3-8% of the frame in the boxed frames

BOXES = {
    14: [[0, 200, 275, 350]],
    20: [[25, 200, 275, 350]],
    32: [[25, 200, 150, 325]],
    34: [[25, 200, 150, 325], [450, 275, 600, 400]],
}

VIDEO_PATH = "/content/drive/MyDrive/egg_project/video/input.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)

def get_frame(sec):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(sec * fps))
    ok, frame = cap.read()
    return frame

def oval_proxy(h, w, boxes):
    m = np.zeros((h, w), dtype=np.uint8)
    for b in boxes:
        centre = (int((b[0] + b[2]) / 2), int((b[1] + b[3]) / 2))
        axes = (int((b[2] - b[0]) / 2), int((b[3] - b[1]) / 2))
        cv2.ellipse(m, centre, axes, 0, 0, 360, 255, -1)
    return m > 0

def drop_big_blobs(mask_bool, max_frac):
    h, w = mask_bool.shape
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask_bool.astype(np.uint8))
    out = np.zeros((h, w), dtype=bool)
    for k in range(1, n):
        if stats[k, cv2.CC_STAT_AREA] <= max_frac * h * w:
            out[labels == k] = True
    return out

def segment_video2(frame, params):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = (hsv[:, :, 1] <= params["sat_max"]) & (hsv[:, :, 2] >= params["val_min"])
    mask[:ROI_TOP, :] = False                      # ignore above the tracking zone
    mask[ROI_BOTTOM:, :] = False                   # ignore below the tracking zone
    mask = clean_mask(mask, 9, 0)                  # opening, same as E3
    mask = filter_blobs(mask, 0.02, 1.0, 3.0)      # min size + aspect, same as E6
    return drop_big_blobs(mask, MAX_AREA_FRAC)     # new: remove oversized blobs

# ---- 3. Sweep against the oval proxy on the 4 boxed frames ----
frames = {}
proxies = {}
for sec in BOXES:
    frames[sec] = get_frame(sec)
    proxies[sec] = oval_proxy(frames[sec].shape[0], frames[sec].shape[1], BOXES[sec])

print("Sweep (mean proxy IoU, then IoU per frame " + ", ".join(str(s) + "s" for s in BOXES) + "):")
print("  sat_max val_min  mean")
best_score = -1
best_params = None
for sat_max in [40, 70, 100]:
    for val_min in [120, 140, 170]:
        per = []
        for sec in BOXES:
            pred = segment_video2(frames[sec], {"sat_max": sat_max, "val_min": val_min})
            per.append(compute_metrics(pred, proxies[sec])["iou"])
        score = float(np.mean(per))
        print("  ", sat_max, "    ", val_min, "   ", round(score, 3), "  ", [round(p, 2) for p in per])
        if score > best_score:
            best_score = score
            best_params = {"sat_max": sat_max, "val_min": val_min}
print("Best:", best_params, "-> proxy IoU", round(best_score, 3))

log_row({"trial_id": "V2", "hypothesis": "Eggs stay in a horizontal zone and are smaller than clothes; position and size rules remove clothing that colour cannot.",
         "method_change": "Video only: V1 colour sweep + tracking zone y=160-440 + max blob size 12% of frame; opening and blob filter as E6",
         "parameters": str(best_params) + " zone=" + str(ROI_TOP) + "-" + str(ROI_BOTTOM) + " max_area=" + str(MAX_AREA_FRAC),
         "motivation": "V1 marked shirts and pants.",
         "dev_macro_iou": round(best_score, 4), "dev_macro_dice": "", "dev_macro_precision": "", "dev_macro_recall": "",
         "dev_micro_iou": "", "observations": "Proxy IoU on 4 boxed frames only; zone assumption checked visually on 16 preview frames",
         "decision": "pending visual check", "next_step": "build tracker if the preview looks right"})

# ---- 4. 16-frame preview with the tracking zone drawn in yellow ----
plt.figure(figsize=(20, 12))
count = 0
for sec in range(10, 42, 2):
    frame = get_frame(sec)
    mask = segment_video2(frame, best_params)
    shown = frame.copy()
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8))
    for k in range(1, n):
        comp = (labels == k).astype(np.uint8)
        contours, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(shown, contours, -1, (0, 255, 0), 3)
        cv2.putText(shown, str(k), (int(centroids[k][0]), int(centroids[k][1])),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
    cv2.line(shown, (0, ROI_TOP), (shown.shape[1], ROI_TOP), (0, 255, 255), 2)
    cv2.line(shown, (0, ROI_BOTTOM), (shown.shape[1], ROI_BOTTOM), (0, 255, 255), 2)
    count += 1
    plt.subplot(4, 4, count)
    plt.imshow(cv2.cvtColor(shown, cv2.COLOR_BGR2RGB))
    plt.title(str(sec) + " s  |  blobs: " + str(n - 1), fontsize=11)
    plt.axis("off")
cap.release()
plt.tight_layout()
plt.show()

In [ ]:
# ===== CELL 22: diagnose table blobs (26 s, 30 s) and the missed egg (36 s) =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

for name in ["segment_video2", "clean_mask", "filter_blobs", "drop_big_blobs"]:
    if name not in globals():
        raise RuntimeError("Run Cell 21 first in this session. Missing: " + name)

VIDEO_PATH = "/content/drive/MyDrive/egg_project/video/input.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)

def get_frame(sec):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(sec * fps))
    ok, frame = cap.read()
    return frame

def describe(frame, params, title):
    """Number each blob on the picture and print its numbers."""
    h, w = frame.shape[:2]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = segment_video2(frame, params)
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8))
    shown = frame.copy()
    print("\n" + title + " | blobs:", n - 1)
    print("   id | area% | aspect | solidity | sat | val | centre (x, y) | box (left, top, width, height)")
    for k in range(1, n):
        comp = (labels == k)
        contours, _ = cv2.findContours(comp.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        big = max(contours, key=cv2.contourArea)
        hull_area = cv2.contourArea(cv2.convexHull(big))
        solidity = cv2.contourArea(big) / hull_area if hull_area > 0 else 0
        (cx, cy), (rw, rh), angle = cv2.minAreaRect(big)
        aspect = max(rw, rh) / min(rw, rh) if min(rw, rh) > 0 else 99
        sat = hsv[:, :, 1][comp].mean()
        val = hsv[:, :, 2][comp].mean()
        x, y, bw, bh = stats[k, 0], stats[k, 1], stats[k, 2], stats[k, 3]
        print("   ", k, "|", round(100 * stats[k, cv2.CC_STAT_AREA] / (h * w), 1), "|", round(aspect, 2),
              "|", round(solidity, 2), "|", int(sat), "|", int(val),
              "| (", int(centroids[k][0]), ",", int(centroids[k][1]), ") | (", x, y, bw, bh, ")")
        cv2.drawContours(shown, contours, -1, (0, 255, 0), 3)
        cv2.putText(shown, str(k), (int(centroids[k][0]), int(centroids[k][1])),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 0, 255), 3)
    return shown

V2 = {"sat_max": 100, "val_min": 120}
pics = []
pics.append((describe(get_frame(26), V2, "26 s (V2)"), "26 s"))
pics.append((describe(get_frame(30), V2, "30 s (V2)"), "30 s"))
pics.append((describe(get_frame(36), V2, "36 s (V2)"), "36 s  V2 (one egg missed)"))
pics.append((describe(get_frame(36), {"sat_max": 100, "val_min": 90}, "36 s (brightness limit 90)"), "36 s  brightness limit 90"))
cap.release()

plt.figure(figsize=(20, 11))
for i in range(4):
    plt.subplot(2, 2, i + 1)
    plt.imshow(cv2.cvtColor(pics[i][0], cv2.COLOR_BGR2RGB))
    plt.title(pics[i][1], fontsize=14)
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ===== CELL 23: V3 = bright core + small grow =====
import numpy as np
import cv2
import matplotlib.pyplot as plt

for name in ["clean_mask", "filter_blobs", "compute_metrics", "oval_proxy", "BOXES",
             "ROI_TOP", "ROI_BOTTOM", "log_row", "record_decision"]:
    if name not in globals():
        raise RuntimeError("Run Cells 11, 13, 14, 16 and 21 first in this session. Missing: " + name)

# ---- 1. Record V2 ----
record_decision(
    "V2",
    "Brightness limit is the key setting (proxy IoU 0.20 at 170 vs 0.44 at 120). Zone + size limit removed shirts/pants, but table still marked at 22-34 s (blobs with brightness 111-163, solidity 0.55-0.79) and one egg missed at 36 s.",
    "discard (table still marked)",
    "V3: bright core (eggs are brightness >200, table is dimmer) then small grow")

VIDEO_PATH = "/content/drive/MyDrive/egg_project/video/input.mp4"
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)

def get_frame(sec):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(sec * fps))
    ok, frame = cap.read()
    return frame

def segment_video3(frame, params):
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    sat = hsv[:, :, 1]
    val = hsv[:, :, 2]
    loose = (sat <= 100) & (val >= 120)
    core = (sat <= 60) & (val >= params["core_val"])
    loose[:ROI_TOP, :] = False
    loose[ROI_BOTTOM:, :] = False
    core[:ROI_TOP, :] = False
    core[ROI_BOTTOM:, :] = False
    core = clean_mask(core, 9, 0)                       # opening removes thin bright specks
    if params["grow"] > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (params["grow"], params["grow"]))
        grown = cv2.dilate(core.astype(np.uint8), kernel) > 0
    else:
        grown = core
    mask = grown & loose                                # grow only where the loose rule agrees
    return filter_blobs(mask, 0.01, 1.0, 3.0)

def count_blobs(mask_bool):
    return cv2.connectedComponents(mask_bool.astype(np.uint8))[0] - 1

# ---- 2. How bright are the eggs inside your boxes? ----
frames = {}
proxies = {}
print("Brightness inside the boxed eggs (50th and 90th percentile) per frame:")
for sec in BOXES:
    frames[sec] = get_frame(sec)
    proxies[sec] = oval_proxy(frames[sec].shape[0], frames[sec].shape[1], BOXES[sec])
    hsv = cv2.cvtColor(frames[sec], cv2.COLOR_BGR2HSV)
    v = hsv[:, :, 2][proxies[sec]]
    print("  ", sec, "s | brightness p50 =", int(np.percentile(v, 50)), "| p90 =", int(np.percentile(v, 90)),
          "| median saturation =", int(np.median(hsv[:, :, 1][proxies[sec]])), "| eggs in boxes:", len(BOXES[sec]))

# ---- 3. Sweep: egg-count match first, IoU second ----
print("\nSweep on the 4 boxed frames:")
print("  core_val grow | mean IoU | frames with right egg count (of 4) | blobs per frame")
best = None
best_params = None
for core_val in [170, 190, 210]:
    for grow in [0, 9, 21]:
        params = {"core_val": core_val, "grow": grow}
        ious = []
        matches = 0
        counts = []
        for sec in BOXES:
            pred = segment_video3(frames[sec], params)
            ious.append(compute_metrics(pred, proxies[sec])["iou"])
            c = count_blobs(pred)
            counts.append(c)
            if c == len(BOXES[sec]):
                matches += 1
        mean_iou = float(np.mean(ious))
        print("  ", core_val, "     ", grow, "   |", round(mean_iou, 3), "|", matches, "|", counts)
        key = (matches, mean_iou)
        if best is None or key > best:
            best = key
            best_params = params
print("Best:", best_params, "-> egg-count matches", best[0], "of 4, proxy IoU", round(best[1], 3))

log_row({"trial_id": "V3", "hypothesis": "Eggs have a bright core (brightness >200); table pieces are dimmer, so a core-based rule drops the table.",
         "method_change": "Video only: bright, low-saturation core -> opening -> small dilation -> keep only where loose rule (sat<=100, val>=120) agrees; zone y=160-440; blob filter min 1%, aspect 3.0",
         "parameters": str(best_params), "motivation": "V2 still marked the table (blobs val 111-163, solidity 0.55-0.79) and merged eggs with table.",
         "dev_macro_iou": round(best[1], 4), "dev_macro_dice": "", "dev_macro_precision": "", "dev_macro_recall": "",
         "dev_micro_iou": "", "observations": "4 boxed frames, oval proxy; video has 2 eggs in total",
         "decision": "pending visual check", "next_step": "build tracker"})

# ---- 4. 16-frame preview ----
plt.figure(figsize=(20, 12))
count = 0
for sec in range(10, 42, 2):
    frame = get_frame(sec)
    mask = segment_video3(frame, best_params)
    shown = frame.copy()
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8))
    for k in range(1, n):
        comp = (labels == k).astype(np.uint8)
        contours, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(shown, contours, -1, (0, 255, 0), 3)
        cv2.putText(shown, str(k), (int(centroids[k][0]), int(centroids[k][1])),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
    cv2.line(shown, (0, ROI_TOP), (shown.shape[1], ROI_TOP), (0, 255, 255), 2)
    cv2.line(shown, (0, ROI_BOTTOM), (shown.shape[1], ROI_BOTTOM), (0, 255, 255), 2)
    count += 1
    plt.subplot(4, 4, count)
    plt.imshow(cv2.cvtColor(shown, cv2.COLOR_BGR2RGB))
    plt.title(str(sec) + " s  |  blobs: " + str(n - 1), fontsize=11)
    plt.axis("off")
cap.release()
plt.tight_layout()
plt.show()

In [ ]:
# ===== CELL 24: Tracker + annotated video (FULL run 10-41 s) =====
import os, csv, subprocess, shutil
from collections import Counter
import numpy as np
import cv2
import matplotlib.pyplot as plt

for name in ["segment_video3", "ROI_TOP", "ROI_BOTTOM", "record_decision"]:
    if name not in globals():
        raise RuntimeError("Run Cells 11, 13, 14, 16, 21 and 23 first in this session. Missing: " + name)

# ---- 0. Record V3 as the final video segmentation ----
record_decision(
    "V3",
    "Bright core (sat<=60, val>=170) + opening + 21 px grow, limited to loose rule (sat<=100, val>=120), zone y=160-440. Proxy IoU 0.554 on 4 boxed frames (V2 0.443); right egg count in 3 of 4 frames; 16-frame preview judged good by visual check.",
    "keep, final video segmentation",
    "Build centroid tracker with persistent IDs and render annotated video")

# ---- 1. Settings ----
START_SEC = 10
END_SEC = 41              # full run (egg part of the video)
VIDEO_PARAMS = {"core_val": 170, "grow": 21}
MAX_DIST = 100            # max centroid jump (pixels) between frames
EXTRA_DIST_PER_MISS = 5   # allowed extra distance per frame a track was unseen
MAX_MISSED = 24           # keep a lost track for 24 frames (1 s)
MIN_HITS = 3              # frames needed before a new track gets an ID
IGNORE_NEAR = 40          # ignore new blobs this close to an already matched blob

PROJECT = "/content/drive/MyDrive/egg_project"
VIDEO_PATH = PROJECT + "/video/input.mp4"
tag = str(START_SEC) + "s_to_" + str(END_SEC) + "s"
RAW_OUT = "/content/tracked_raw_" + tag + ".mp4"
H264_OUT = "/content/tracked_" + tag + ".mp4"
DRIVE_OUT = PROJECT + "/video/tracked_" + tag + ".mp4"
LOG_OUT = PROJECT + "/results/tracking_log_" + tag + ".csv"

COLOURS = [(0, 255, 0), (0, 165, 255), (255, 0, 255), (255, 255, 0), (0, 0, 255), (255, 128, 0)]

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
start_frame = int(START_SEC * fps)
end_frame = int(END_SEC * fps)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
writer = cv2.VideoWriter(RAW_OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
print("Processing frames", start_frame, "to", end_frame, "(", END_SEC - START_SEC, "seconds )")

# ---- 2. Detection helper ----
def find_detections(mask):
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8))
    dets = []
    for k in range(1, n):
        comp = (labels == k).astype(np.uint8)
        contours, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        dets.append({"cx": float(centroids[k][0]), "cy": float(centroids[k][1]),
                     "box": (int(stats[k, 0]), int(stats[k, 1]), int(stats[k, 2]), int(stats[k, 3])),
                     "area": int(stats[k, cv2.CC_STAT_AREA]), "contours": contours})
    return dets

# ---- 3. Tracking loop ----
tracks = []        # each track: dict with id (None until confirmed), cx, cy, missed, hits, trail
next_id = 1
log_rows = []
tracked_per_frame = []
detections_per_frame = []
sample_frames = set(np.linspace(start_frame, end_frame - 1, 8).astype(int).tolist())
samples = []

for frame_no in range(start_frame, end_frame):
    ok, frame = cap.read()
    if not ok:
        break
    mask = segment_video3(frame, VIDEO_PARAMS)
    dets = find_detections(mask)
    detections_per_frame.append(len(dets))

    # (a) match tracks and detections: closest pairs first
    pairs = []
    for ti in range(len(tracks)):
        for di in range(len(dets)):
            d = np.hypot(tracks[ti]["cx"] - dets[di]["cx"], tracks[ti]["cy"] - dets[di]["cy"])
            limit = MAX_DIST + EXTRA_DIST_PER_MISS * tracks[ti]["missed"]
            if d <= limit:
                pairs.append((d, ti, di))
    pairs.sort()
    used_tracks = set()
    used_dets = set()
    for d, ti, di in pairs:
        if ti in used_tracks or di in used_dets:
            continue
        used_tracks.add(ti)
        used_dets.add(di)
        t = tracks[ti]
        t["cx"] = dets[di]["cx"]
        t["cy"] = dets[di]["cy"]
        t["det"] = dets[di]
        t["missed"] = 0
        t["hits"] = t["hits"] + 1
        t["trail"].append((int(t["cx"]), int(t["cy"])))
        if t["id"] is None and t["hits"] >= MIN_HITS:
            t["id"] = next_id
            next_id = next_id + 1

    # (b) tracks with no detection: count a miss
    for ti in range(len(tracks)):
        if ti not in used_tracks:
            tracks[ti]["missed"] = tracks[ti]["missed"] + 1
            tracks[ti]["det"] = None

    # (c) new tracks from unmatched detections (skip fragments next to a matched blob)
    for di in range(len(dets)):
        if di in used_dets:
            continue
        too_close = False
        for dj in used_dets:
            if np.hypot(dets[di]["cx"] - dets[dj]["cx"], dets[di]["cy"] - dets[dj]["cy"]) < IGNORE_NEAR:
                too_close = True
        if too_close:
            continue
        first_id = None
        if frame_no == start_frame:          # eggs already visible in the first frame get IDs at once
            first_id = next_id
            next_id = next_id + 1
        tracks.append({"id": first_id, "cx": dets[di]["cx"], "cy": dets[di]["cy"], "missed": 0, "hits": 1,
                       "trail": [(int(dets[di]["cx"]), int(dets[di]["cy"]))], "det": dets[di]})

    # (d) delete old tracks
    keep = []
    for t in tracks:
        if t["id"] is None and t["missed"] > 2:
            continue
        if t["id"] is not None and t["missed"] > MAX_MISSED:
            continue
        keep.append(t)
    tracks = keep

    # (e) draw and log tracks that are confirmed and seen in this frame
    shown = frame.copy()
    visible = 0
    for t in tracks:
        if t["id"] is None or t["missed"] != 0 or t["det"] is None:
            continue
        visible = visible + 1
        colour = COLOURS[(t["id"] - 1) % len(COLOURS)]
        x, y, bw, bh = t["det"]["box"]
        cv2.drawContours(shown, t["det"]["contours"], -1, colour, 2)
        cv2.rectangle(shown, (x, y), (x + bw, y + bh), colour, 2)
        cv2.circle(shown, (int(t["cx"]), int(t["cy"])), 5, colour, -1)
        if len(t["trail"]) > 1:
            cv2.polylines(shown, [np.array(t["trail"][-40:], dtype=np.int32)], False, colour, 2)
        cv2.putText(shown, "egg " + str(t["id"]), (x, max(20, y - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, colour, 2)
        log_rows.append([frame_no, round(frame_no / fps, 2), t["id"], int(t["cx"]), int(t["cy"]), x, y, bw, bh, t["det"]["area"]])
    tracked_per_frame.append(visible)
    cv2.putText(shown, "t = %.1f s   eggs tracked: %d" % (frame_no / fps, visible), (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
    writer.write(shown)
    if frame_no in sample_frames:
        samples.append((frame_no / fps, cv2.cvtColor(shown, cv2.COLOR_BGR2RGB)))

cap.release()
writer.release()

# ---- 4. Convert to H.264 (plays in Google Drive) and copy to Drive ----
final_file = RAW_OUT
result = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", RAW_OUT, "-vcodec", "libx264",
                         "-pix_fmt", "yuv420p", H264_OUT], capture_output=True, text=True)
if result.returncode == 0 and os.path.exists(H264_OUT):
    final_file = H264_OUT
    print("Re-encoded to H.264.")
else:
    print("ffmpeg re-encode failed, using the raw mp4v file. Message:", result.stderr[-200:])
shutil.copy(final_file, DRIVE_OUT)
print("Video saved to Drive:", DRIVE_OUT, "| size (MB):", round(os.path.getsize(DRIVE_OUT) / 1e6, 1))

with open(LOG_OUT, "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(["frame", "time_s", "id", "cx", "cy", "box_x", "box_y", "box_w", "box_h", "area_px"])
    wr.writerows(log_rows)
print("Tracking table saved:", LOG_OUT)

# ---- 5. Statistics ----
n_frames = len(tracked_per_frame)
print("\nFrames processed:", n_frames)
print("Raw blobs per frame:", dict(sorted(Counter(detections_per_frame).items())))
print("Eggs tracked per frame:", dict(sorted(Counter(tracked_per_frame).items())))
ids = sorted(set(r[2] for r in log_rows))
print("Number of IDs created:", len(ids), "(the video has 2 eggs in total)")
for i in ids:
    frames_i = [r[0] for r in log_rows if r[2] == i]
    print("   egg", i, "| frames seen:", len(frames_i), "| first", round(min(frames_i) / fps, 1), "s | last", round(max(frames_i) / fps, 1), "s")

# ---- 6. Charts ----
plt.figure(figsize=(14, 3))
times = [(start_frame + i) / fps for i in range(n_frames)]
plt.plot(times, tracked_per_frame, label="tracked (with ID)")
plt.plot(times, detections_per_frame, label="raw blobs", alpha=0.5)
plt.xlabel("time (s)"); plt.ylabel("eggs"); plt.title("Eggs per frame"); plt.legend()
plt.show()

plt.figure(figsize=(20, 9))
for i in range(len(samples)):
    plt.subplot(2, 4, i + 1)
    plt.imshow(samples[i][1])
    plt.title("%.1f s" % samples[i][0])
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ===== CELL 25: build the repo folder on Drive (works on a fresh CPU session) =====
import os, shutil, glob, json
from importlib.metadata import version, PackageNotFoundError

from google.colab import drive
drive.mount("/content/drive")

PROJECT = "/content/drive/MyDrive/egg_project"
REPO = PROJECT + "/repo"

# 1. Folder structure
for sub in ["src", "notebooks", "configs", "experiments", "results/plots", "results/reference_masks_zip", "report"]:
    os.makedirs(REPO + "/" + sub, exist_ok=True)

def copy_if_exists(src, dst_folder):
    if os.path.exists(src):
        shutil.copy(src, REPO + "/" + dst_folder + "/")
        print("copied:", os.path.basename(src))
    else:
        print("MISSING (skipped):", src)

# 2. Copy real results
copy_if_exists(PROJECT + "/experiments/experiment_log.csv", "experiments")
copy_if_exists(PROJECT + "/experiments/split_manifest.csv", "experiments")
copy_if_exists(PROJECT + "/results/reference_log_final.csv", "results")
copy_if_exists(PROJECT + "/results/eval_metrics_per_image.csv", "results")
copy_if_exists(PROJECT + "/results/eval_metrics_summary.csv", "results")
copy_if_exists(PROJECT + "/results/tracking_log_10s_to_41s.csv", "results")
for p in glob.glob(PROJECT + "/results/plots/*.png"):
    copy_if_exists(p, "results/plots")

# 3. Zip the final reference masks (one file instead of 110)
mask_dir = PROJECT + "/results/reference_masks"
if os.path.exists(mask_dir):
    shutil.make_archive(REPO + "/results/reference_masks_zip/reference_masks", "zip", mask_dir)
    print("zipped: reference_masks.zip")
else:
    print("MISSING (skipped):", mask_dir)

# 4. Frozen parameters (copied from the experiments we ran)
params = {
    "split": {"seed": 42, "eval_fraction": 0.2, "method": "dHash (9x8), Hamming threshold 10, group-aware"},
    "sam_reference": {
        "model": "SAM ViT-B (segment-anything)",
        "points_per_side": 32, "pred_iou_thresh": 0.88, "stability_score_thresh": 0.92, "min_mask_region_area": 200,
        "egg_rule": {"min_area_frac": 0.02, "max_area_frac": 0.90, "min_solidity": 0.88, "max_aspect": 2.5,
                     "max_edge_cover": 0.20, "max_saturation": 70, "min_value_strict": 170, "min_value_relaxed": 155},
        "manual_box_images": 17, "total_images": 110},
    "classical_final_E6": {"sat_max": 40, "val_min": 170, "opening_kernel": 9, "closing_kernel": 0,
                           "min_area_frac": 0.02, "max_edge": 1.0, "max_aspect": 3.0},
    "video_V3": {"core_sat_max": 60, "core_val_min": 170, "grow_kernel": 21, "loose_sat_max": 100, "loose_val_min": 120,
                 "zone_y": [160, 440], "opening_kernel": 9, "min_area_frac": 0.01, "max_aspect": 3.0,
                 "tracked_seconds": [10, 41]},
    "tracker": {"max_dist_px": 100, "extra_dist_per_missed_frame": 5, "max_missed_frames": 24,
                "min_hits_for_new_id": 3, "ignore_new_blob_within_px": 40}
}
with open(REPO + "/configs/final_params.json", "w") as f:
    json.dump(params, f, indent=2)
print("wrote configs/final_params.json")

# 5. requirements.txt (pinned where the package is installed in this session)
lines = []
for pkg in ["numpy", "opencv-python", "matplotlib", "torch", "torchvision", "roboflow", "yt-dlp"]:
    try:
        lines.append(pkg + "==" + version(pkg))
    except PackageNotFoundError:
        if pkg == "opencv-python":
            try:
                lines.append("opencv-python-headless==" + version("opencv-python-headless"))
            except PackageNotFoundError:
                lines.append("opencv-python")
        else:
            lines.append(pkg)
lines.append("segment-anything @ git+https://github.com/facebookresearch/segment-anything.git")
with open(REPO + "/requirements.txt", "w") as f:
    f.write("\n".join(lines) + "\n")
print("\nrequirements.txt:")
print("\n".join(lines))

# 6. Show everything and the total size
print("\nREPO CONTENTS:")
total = 0
for root, dirs, files in os.walk(REPO):
    for name in files:
        path = os.path.join(root, name)
        size = os.path.getsize(path)
        total += size
        print("  ", path.replace(REPO + "/", ""), "-", round(size / 1e6, 2), "MB")
print("Total size (MB):", round(total / 1e6, 1))